# Assignment 04 — Miền CIFAR-10 · Notebook 02: Đối chiếu PyTorch và TensorFlow/Keras

**Học phần:** Phát triển các Hệ thống Thông minh — Học viện Công nghệ Bưu chính Viễn thông
**Sinh viên:** Nguyễn Duy Nghĩa · **Mã sinh viên:** B23DCCN600 · **Lớp:** D23CTPM01
**Giảng viên hướng dẫn:** PGS.TS Trần Đình Quế
**Học kỳ:** Học kỳ 1 năm học 2026 – 2027

---

## Mục tiêu của notebook

Notebook 01 đã chứng minh rằng một mạng tích chập viết tay bằng NumPy hoạt động đúng về mặt toán
học. Notebook này trả lời câu hỏi tiếp theo: **khi được tự do dùng framework, ta đạt tới đâu trên
CIFAR-10, và khoảng cách so với cài đặt thủ công đến từ đâu?**

Bốn việc được thực hiện:

1. Dựng cùng một kiến trúc sâu ba khối bằng **PyTorch** và bằng **Keras**, đối chiếu số tham số
   để chắc chắn hai mạng thực sự tương đương.
2. Lưu ba hiện vật bàn giao cho notebook `mlp_vs_cnn`: trọng số `cifar10_cnn_pytorch.pt`, định
   nghĩa mô-đun `cifar10_cnn_def.py`, hằng số chuẩn hóa `cifar10_preproc.json`. Mô hình PyTorch
   **bắt buộc** có phương thức `extract_features(x)` trả về véc-tơ ẩn 128 chiều để phân tích PCA.
3. So sánh ba cách cài đặt trên cùng 10 000 ảnh kiểm thử: NumPy thuần, PyTorch, Keras.
4. Phân tích sâu mô hình tốt nhất: độ chính xác từng lớp và tám ảnh sai với độ tin cậy cao nhất.

**Năm hình bắt buộc sinh ra ở đây:** `fig_cifar10_framework_curves.png`,
`fig_cifar10_framework_confusion.png`, `fig_cifar10_3way_benchmark.png`,
`fig_cifar10_per_class_accuracy.png`, `fig_cifar10_high_conf_errors.png`. Notebook cũng ghi tệp
metrics tổng hợp `reports/metrics_cifar10.json` theo lược đồ đa lớp ở mục 5.3 hợp đồng tích hợp.

## Quy mô huấn luyện: đầy đủ, không còn lấy mẫu con

Bản trước của notebook này chạy trên một tập con nhỏ vì mọi thứ đều nằm trên CPU. Điều kiện đó đã
thay đổi: PyTorch hiện có CUDA và card RTX 4060 Laptop xử lý một epoch trên 40 000 ảnh trong vài
giây. Vì vậy hai mô hình framework được huấn luyện trên **toàn bộ 40 000 ảnh train, 10 000 ảnh
validation, 25 epoch, cỡ lô 128**, rồi đánh giá trên **trọn vẹn 10 000 ảnh kiểm thử**.

Hai mô hình NumPy ở notebook 01 vẫn dùng tập con 10 000 / 2 500 ảnh. Đó **không** phải hệ quả của
giới hạn phần cứng mà là bản chất của bài tập: đề bài yêu cầu hiện thực bằng NumPy thuần, và NumPy
chạy trên CPU. Có GPU cũng không đổi được điều đó.

## Kiến trúc: Conv-BN-ReLU-MaxPool-Dropout ba khối

```
Đầu vào (3, 32, 32)
  Khối 1: Conv(3->32, 3x3, same) -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout(0,25)   ->  (32, 16, 16)
  Khối 2: Conv(32->64, 3x3, same) -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout(0,25)  ->  (64,  8,  8)
  Khối 3: Conv(64->64, 3x3, same) -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout(0,25)  ->  (64,  4,  4)
  Flatten (1024) -> Dense(128) -> ReLU -> Dropout(0,5) -> Dense(10)
```

Ba thành phần mà cài đặt NumPy ở notebook 01 **không có**, và đó chính là nguồn gốc chính của
chênh lệch kết quả:

- **Batch Normalization.** Chuẩn hóa kích hoạt theo từng lô rồi học lại hệ số tỉ lệ và độ dời:
  $\hat{h} = \gamma \frac{h - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$. Nó làm phẳng mặt mất
  mát nên cho phép learning rate lớn hơn và hội tụ nhanh hơn đáng kể.
- **Dropout.** Tắt ngẫu nhiên một phần kích hoạt khi huấn luyện, buộc mạng không phụ thuộc vào
  một vài nơ-ron riêng lẻ. Notebook 01 đã cho thấy mạng NumPy quá khớp rõ rệt; dropout là câu trả
  lời trực tiếp cho vấn đề đó.
- **Khối thứ ba.** Thêm một tầng tích chập nữa mở rộng trường tiếp nhận (receptive field) tới mức
  bao phủ gần trọn ảnh $32 \times 32$, cho phép mạng nhìn thấy hình dáng tổng thể chứ không chỉ
  các mảnh cục bộ.

## 1. Nhập thư viện, cố định hạt giống và xác lập thiết bị

Ba mô hình của miền CIFAR-10 chạy trên **ba cấu hình phần cứng khác nhau**, và đây là điều phải
nói thẳng trước khi đọc bất kỳ con số thời gian nào trong báo cáo:

| Mô hình | Thư viện | Thiết bị | Nguyên nhân |
|---|---|---|---|
| `numpy_baseline`, `numpy_improved` | NumPy | **CPU** | NumPy là thư viện tính toán trên CPU; đây là bản chất của đề bài |
| `pytorch` | PyTorch 2.13.0+cu126 | **GPU** (RTX 4060 Laptop, CUDA 12.6) | có hỗ trợ CUDA đầy đủ |
| `tensorflow` | TensorFlow 2.21.0 / Keras 3.15.1 | **CPU** | TensorFlow từ bản 2.11 đã bỏ hỗ trợ GPU native trên Windows |

Hệ quả bắt buộc phải rút ra: **cột thời gian trong mọi bảng của notebook này là phép so sánh phần
cứng chứ không phải phép so sánh khung thư viện.** Một phát biểu dạng "PyTorch nhanh hơn Keras $N$
lần" dựa trên các con số ở đây chỉ nói rằng GPU nhanh hơn CPU, điều đã biết trước khi thí nghiệm
bắt đầu. Báo cáo vì thế ghi tên thiết bị bên cạnh mọi con số thời gian và **không** kết luận khung
nào nhanh hơn khung nào. Muốn so sánh hai khung về tốc độ thì phải chạy cả hai trên cùng một loại
phần cứng, việc mà cấu hình Windows hiện tại không cho phép.

Ngược lại, **mọi chỉ số chất lượng vẫn so sánh được bình thường**: accuracy, precision, recall và
F1 được tính từ nhãn dự đoán trên cùng một tập kiểm thử, không phụ thuộc vào thiết bị đã sinh ra
nhãn đó. Hai mạng cùng kiến trúc, cùng siêu tham số, cùng số epoch, cùng dữ liệu thì phải cho chất
lượng tương đương dù chạy trên hai loại phần cứng. Kiểm chứng chính điều đó là mục đích của mục 7.

### Đo thời gian trên GPU: vì sao bắt buộc phải đồng bộ hóa

Lời gọi CUDA là **bất đồng bộ**. Khi trình thông dịch Python thực hiện xong dòng lệnh phóng một
nhân tính toán, GPU thường vẫn chưa tính xong: hàm trả về ngay sau khi lệnh được xếp vào hàng đợi
của thiết bị. Đọc `time.time()` ngay tại thời điểm đó sẽ thu được **thời gian xếp hàng lệnh** chứ
không phải thời gian tính toán, và con số đó có thể nhỏ hơn thời gian thật hàng chục lần — một
cách vô tình thổi phồng ưu thế của GPU.

Mọi phép đo thời gian trong notebook này vì thế đều bọc `torch.cuda.synchronize()` ở **cả hai
đầu**, buộc CPU chờ cho tới khi GPU hoàn tất toàn bộ hàng đợi rồi mới đọc đồng hồ.

In [1]:
import os, json, time, copy, sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             classification_report)

import torch
import torch.nn as nn

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
import keras
from keras import layers

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)          # seed này phủ cả CPU lẫn mọi thiết bị CUDA
tf.random.set_seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)
torch.use_deterministic_algorithms(False)

# --- Thiết bị: PyTorch BẮT BUỘC chạy GPU theo mục 1 hợp đồng tích hợp ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'Phai chay tren GPU; kiem tra lai venv'
print('Thiet bi:', torch.cuda.get_device_name(0))

plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH  = '../data/cifar10.npz'
FIG_DIR    = '../reports/figures'
REP_DIR    = '../reports'
MODEL_DIR  = '../models'
for d in (FIG_DIR, REP_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

CLASS_EN = ['airplane', 'automobile', 'bird', 'cat', 'deer',
            'dog', 'frog', 'horse', 'ship', 'truck']
CLASS_VI = ['máy bay', 'ô tô', 'chim', 'mèo', 'hươu',
            'chó', 'ếch', 'ngựa', 'tàu thủy', 'xe tải']

EPOCHS_FW = 25
BATCH_FW  = 128

# Nhãn thiết bị dùng lại ở mọi bảng, mọi tiêu đề hình và mọi khóa JSON
GPU_NAME   = torch.cuda.get_device_name(0)
DEV_PT     = 'cuda'
DEV_TF     = 'cpu'
LBL_PT     = f'GPU · {GPU_NAME}'
LBL_TF     = 'CPU'

_tf_gpus = tf.config.list_physical_devices('GPU')

print()
print('NumPy      :', np.__version__)
print('PyTorch    :', torch.__version__, '| CUDA:', torch.version.cuda,
      '| khả dụng:', torch.cuda.is_available())
print(f'  -> thiết bị PyTorch : {DEVICE} ({GPU_NAME}, '
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB, '
      f'compute capability {".".join(map(str, torch.cuda.get_device_capability(0)))})')
print('TensorFlow :', tf.__version__, '| Keras:', keras.__version__)
print(f'  -> GPU mà TensorFlow nhìn thấy: {_tf_gpus}')
print('  -> TensorFlow từ bản 2.11 KHÔNG còn hỗ trợ GPU native trên Windows, nên Keras chạy CPU.')
print('     Đây là hành vi đúng như tài liệu của TensorFlow, không phải lỗi cấu hình.')
print()
print(f'Cấu hình huấn luyện framework: {EPOCHS_FW} epoch, batch {BATCH_FW}, quy mô đầy đủ')
print(f'Nhãn thiết bị dùng trong báo cáo: PyTorch = "{LBL_PT}" | Keras = "{LBL_TF}"')

Thiet bi: NVIDIA GeForce RTX 4060 Laptop GPU



NumPy      : 2.5.1
PyTorch    : 2.13.0+cu126 | CUDA: 12.6 | khả dụng: True
  -> thiết bị PyTorch : cuda (NVIDIA GeForce RTX 4060 Laptop GPU, 8.6 GB, compute capability 8.9)
TensorFlow : 2.21.0 | Keras: 3.15.1
  -> GPU mà TensorFlow nhìn thấy: []
  -> TensorFlow từ bản 2.11 KHÔNG còn hỗ trợ GPU native trên Windows, nên Keras chạy CPU.
     Đây là hành vi đúng như tài liệu của TensorFlow, không phải lỗi cấu hình.

Cấu hình huấn luyện framework: 25 epoch, batch 128, quy mô đầy đủ
Nhãn thiết bị dùng trong báo cáo: PyTorch = "GPU · NVIDIA GeForce RTX 4060 Laptop GPU" | Keras = "CPU"


## 2. Nạp dữ liệu và tiền xử lý

Phép chia và phép chuẩn hóa lặp lại **chính xác** những gì notebook 01 đã làm: cùng
`random_state=42`, cùng `stratify`, cùng hằng số theo kênh học từ nhánh train. Nhờ vậy ba mô hình
nhìn thấy cùng một tập kiểm thử đã qua cùng một phép biến đổi, và phép so sánh cuối notebook là
công bằng.

Khác biệt duy nhất: hai mô hình framework dùng **toàn bộ 40 000 ảnh train và 10 000 ảnh
validation**, trong khi hai mô hình NumPy ở notebook 01 dùng tập con phân tầng 10 000 / 2 500 ảnh.
Cỡ tập con đó được ghi vào khóa `numpy_subset` của tệp metrics, vào phần `notes`, và vào tiêu đề
của mọi hình có mặt mô hình NumPy, để người đọc không bao giờ nhìn thấy con số của NumPy tách rời
khỏi điều kiện tạo ra nó.

Một lưu ý về bộ nhớ, vì nó quyết định cách tổ chức vòng lặp huấn luyện bên dưới: ba tập dữ liệu ở
dạng `float32` chiếm khoảng 737 MB, nhỏ hơn nhiều so với 8,6 GB bộ nhớ của card đồ họa. Nhờ vậy
toàn bộ dữ liệu có thể nằm thường trú trên GPU trong suốt quá trình huấn luyện, và mỗi epoch không
phải trả chi phí chuyển dữ liệu qua bus PCIe.

In [2]:
assert os.path.exists(DATA_PATH), f'Không tìm thấy {DATA_PATH}'
_d = np.load(DATA_PATH)
x_train_raw, y_train_raw = _d['x_train'], _d['y_train'].astype(np.int64)
x_test_raw,  y_test_raw  = _d['x_test'],  _d['y_test'].astype(np.int64)

idx_all = np.arange(len(x_train_raw))
idx_tr, idx_va = train_test_split(idx_all, test_size=0.2,
                                  stratify=y_train_raw, random_state=RANDOM_SEED)

# Tích lũy BẮT BUỘC ở float64 (xem notebook 00: cộng dồn ở float32 làm trung bình bão hòa
# và trả về ba giá trị trùng nhau, một hiện vật làm tròn chứ không phải thống kê thật).
_tr_u8 = x_train_raw[idx_tr]
MEAN_C = np.array([_tr_u8[:, :, :, c].mean(dtype=np.float64) / 255.0
                   for c in range(3)], dtype=np.float32)
STD_C  = np.array([_tr_u8[:, :, :, c].std(dtype=np.float64) / 255.0
                   for c in range(3)], dtype=np.float32)
del _tr_u8


def preprocess(x_u8):
    '''uint8 (N,32,32,3) HWC -> float32 (N,3,32,32) CHW đã chuẩn hóa theo từng kênh.'''
    x = x_u8.astype(np.float32) / 255.0
    x = (x - MEAN_C) / STD_C
    return np.ascontiguousarray(x.transpose(0, 3, 1, 2))


X_tr, y_tr = preprocess(x_train_raw[idx_tr]), y_train_raw[idx_tr]
X_va, y_va = preprocess(x_train_raw[idx_va]), y_train_raw[idx_va]
X_te, y_te = preprocess(x_test_raw),          y_test_raw

print(f'MEAN theo kênh (R, G, B) = {[round(float(v), 10) for v in MEAN_C]}')
print(f'STD  theo kênh (R, G, B) = {[round(float(v), 10) for v in STD_C]}')
print()
print(f'Train      : {X_tr.shape}  phân phối lớp {np.bincount(y_tr, minlength=10).tolist()}')
print(f'Validation : {X_va.shape}  phân phối lớp {np.bincount(y_va, minlength=10).tolist()}')
print(f'Test       : {X_te.shape}  phân phối lớp {np.bincount(y_te, minlength=10).tolist()}')
print(f'Khoảng giá trị sau chuẩn hóa: [{X_tr.min():.4f}, {X_tr.max():.4f}]')
print(f'Bộ nhớ chiếm dụng: {(X_tr.nbytes + X_va.nbytes + X_te.nbytes)/1e6:.0f} MB (float32)')

preproc = {
    'description': 'Hằng số chuẩn hóa CIFAR-10 theo từng kênh màu, học từ 40000 ảnh nhánh '
                   'train (train_test_split test_size=0.2, stratify=y, random_state=42).',
    'mean': [float(v) for v in MEAN_C],
    'std':  [float(v) for v in STD_C],
    'mean_per_channel': [float(v) for v in MEAN_C],
    'std_per_channel':  [float(v) for v in STD_C],
    'channel_order': ['R', 'G', 'B'],
    'scale': 255.0,
    'formula': 'x_norm = (x_uint8 / 255.0 - mean_c) / std_c, áp dụng riêng cho từng kênh',
    'input_layout': 'NCHW (N, 3, 32, 32) cho PyTorch; dữ liệu gốc trong .npz là NHWC',
    'n_train': int(len(idx_tr)), 'n_val': int(len(idx_va)), 'n_test': int(len(X_te)),
    'n_classes': 10, 'class_names_en': CLASS_EN, 'class_names_vi': CLASS_VI,
    'random_seed': RANDOM_SEED,
}
with open(os.path.join(MODEL_DIR, 'cifar10_preproc.json'), 'w', encoding='utf-8') as f:
    json.dump(preproc, f, ensure_ascii=False, indent=2)
print()
print('Đã lưu', os.path.join(MODEL_DIR, 'cifar10_preproc.json'))

MEAN theo kênh (R, G, B) = [0.4910935163, 0.4821471274, 0.4465753734]
STD  theo kênh (R, G, B) = [0.247007668, 0.2435435355, 0.2616426051]

Train      : (40000, 3, 32, 32)  phân phối lớp [4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000, 4000]
Validation : (10000, 3, 32, 32)  phân phối lớp [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
Test       : (10000, 3, 32, 32)  phân phối lớp [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
Khoảng giá trị sau chuẩn hóa: [-1.9882, 2.1263]
Bộ nhớ chiếm dụng: 737 MB (float32)

Đã lưu ../models\cifar10_preproc.json


## 3. Định nghĩa mô hình PyTorch trong một mô-đun độc lập

Định nghĩa mạng được ghi ra tệp `../models/cifar10_cnn_def.py` thay vì chỉ khai báo trong
notebook. Lý do rất thực tế: notebook `mlp_vs_cnn` cần nạp lại trọng số đã huấn luyện để trích
véc-tơ ẩn cho phân tích PCA, mà `torch.load` trên một `state_dict` đòi hỏi lớp mô hình phải có sẵn
ở phía người nạp. Một tệp `.py` thuần, không phụ thuộc notebook, là cách bàn giao sạch nhất.

Phương thức `extract_features(x)` trả về đầu ra 128 chiều của tầng `Dense(128)` **sau ReLU nhưng
trước Dropout và trước tầng phân loại cuối**. Đây là biểu diễn mà mạng thực sự dùng để ra quyết
định, nên là đối tượng phù hợp nhất cho phân tích không gian ẩn.

In [3]:
%%writefile ../models/cifar10_cnn_def.py
"""Định nghĩa CNN 2D cho CIFAR-10 (Assignment 04, miền cifar10).

Mô-đun độc lập để notebook khác (mlp_vs_cnn) nạp lại trọng số đã huấn luyện tại
models/cifar10_cnn_pytorch.pt và trích véc-tơ ẩn 128 chiều bằng extract_features().

Tiền xử lý bắt buộc khi suy luận (xem models/cifar10_preproc.json):
    x = x_uint8 / 255.0                      # ảnh gốc NHWC (N, 32, 32, 3)
    x = (x - mean_c) / std_c                 # mean_c, std_c là ba số cho ba kênh R, G, B
    x = x.transpose(0, 3, 1, 2)              # -> NCHW (N, 3, 32, 32)

Kiến trúc (theo mục 4 hợp đồng tích hợp, biến thể CIFAR-10 ba khối 32 -> 64 -> 64):
    Conv-BN-ReLU-MaxPool-Dropout x3 -> Flatten -> Dense(128) -> ReLU -> Dropout -> Dense(10)

Cách dùng:
    from cifar10_cnn_def import Cifar10CNN
    model = Cifar10CNN()
    model.load_state_dict(torch.load('cifar10_cnn_pytorch.pt', map_location='cpu'))
    model.eval()
    with torch.no_grad():
        z = model.extract_features(x)        # (N, 128)
"""

import torch
import torch.nn as nn


class Cifar10CNN(nn.Module):
    """CNN 2D ba khối phân cấp cho ảnh màu CIFAR-10."""

    FEATURE_DIM = 128          # số chiều véc-tơ ẩn mà extract_features trả về

    def __init__(self, n_classes: int = 10, p_conv: float = 0.25, p_fc: float = 0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 32 x 32 -> 16 x 16
            nn.Dropout(p_conv),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 16 x 16 -> 8 x 8
            nn.Dropout(p_conv),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                      # 8 x 8 -> 4 x 4
            nn.Dropout(p_conv),
        )
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 4 * 4, self.FEATURE_DIM)
        self.relu_fc = nn.ReLU(inplace=True)
        self.drop_fc = nn.Dropout(p_fc)
        self.fc2 = nn.Linear(self.FEATURE_DIM, n_classes)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """Trả về véc-tơ ẩn 128 chiều ở tầng áp chót, dùng cho phân tích PCA.

        Tham số
        -------
        x : Tensor (N, 3, 32, 32) đã chuẩn hóa theo models/cifar10_preproc.json

        Trả về
        ------
        Tensor (N, 128) sau Dense(128) và ReLU, TRƯỚC Dropout và tầng phân loại.
        """
        h = self.block1(x)
        h = self.block2(h)
        h = self.block3(h)
        h = self.flatten(h)
        return self.relu_fc(self.fc1(h))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Trả về logit chưa qua softmax, (N, 10)."""
        return self.fc2(self.drop_fc(self.extract_features(x)))

Overwriting ../models/cifar10_cnn_def.py


In [4]:
sys.path.insert(0, os.path.abspath(MODEL_DIR))
import importlib
import cifar10_cnn_def
importlib.reload(cifar10_cnn_def)
from cifar10_cnn_def import Cifar10CNN

torch.manual_seed(RANDOM_SEED)
model_pt = Cifar10CNN().to(DEVICE)
n_pt = sum(p.numel() for p in model_pt.parameters() if p.requires_grad)
print(model_pt)
print()
print(f'Tổng tham số học được (PyTorch): {n_pt:,}')
for name, mod in [('block1', model_pt.block1), ('block2', model_pt.block2),
                  ('block3', model_pt.block3), ('fc1', model_pt.fc1), ('fc2', model_pt.fc2)]:
    print(f'  {name:7s}: {sum(p.numel() for p in mod.parameters()):>9,} tham số')
print()
print(f'Mô hình nằm trên thiết bị: {next(model_pt.parameters()).device}')

# Kiểm tra luồng kích thước và hợp đồng của extract_features, chạy ngay trên GPU
with torch.no_grad():
    _x = torch.from_numpy(X_te[:4]).to(DEVICE)
    _h1 = model_pt.block1(_x); _h2 = model_pt.block2(_h1); _h3 = model_pt.block3(_h2)
    _f = model_pt.extract_features(_x)
    _o = model_pt(_x)
print()
print(f'Luồng kích thước: {tuple(_x.shape)} -> khối 1 {tuple(_h1.shape)} '
      f'-> khối 2 {tuple(_h2.shape)} -> khối 3 {tuple(_h3.shape)}')
print(f'extract_features: {tuple(_x.shape)} -> {tuple(_f.shape)}  '
      f'| đúng hợp đồng 128 chiều: {tuple(_f.shape) == (4, 128)}')
print(f'forward         : {tuple(_x.shape)} -> {tuple(_o.shape)}  '
      f'| đúng 10 logit: {tuple(_o.shape) == (4, 10)}')
print(f'Tensor trung gian nằm trên: {_f.device}')
print(f'Véc-tơ ẩn sau ReLU nên không âm: giá trị nhỏ nhất = {float(_f.min()):.6f}')

Cifar10CNN(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.25, inplace=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.25, inplace=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool


Luồng kích thước: (4, 3, 32, 32) -> khối 1 (4, 32, 16, 16) -> khối 2 (4, 64, 8, 8) -> khối 3 (4, 64, 4, 4)
extract_features: (4, 3, 32, 32) -> (4, 128)  | đúng hợp đồng 128 chiều: True
forward         : (4, 3, 32, 32) -> (4, 10)  | đúng 10 logit: True
Tensor trung gian nằm trên: cuda:0
Véc-tơ ẩn sau ReLU nên không âm: giá trị nhỏ nhất = 0.000000


**Diễn giải.** Luồng kích thước đúng như thiết kế: ba lần max pooling đưa ảnh từ
$32 \times 32$ xuống $4 \times 4$ trong khi số kênh tăng từ 3 lên 64. Đây là nguyên tắc "đánh đổi
độ phân giải không gian lấy chiều sâu ngữ nghĩa" của mọi kiến trúc CNN phân cấp: các tầng đầu mô
tả *ở đâu có cạnh gì*, các tầng sau mô tả *có những bộ phận nào*, và vị trí chính xác dần trở nên
không quan trọng.

Phân rã tham số cho thấy tầng `fc1` vẫn chiếm phần lớn trọng số, nhưng tỉ trọng đã dễ chịu hơn
nhiều so với mạng NumPy ở notebook 01, vì ba lần pooling đã nén véc-tơ phẳng xuống còn 1 024
chiều. Véc-tơ ẩn có giá trị nhỏ nhất bằng 0 đúng như kỳ vọng sau ReLU.

## 4. Huấn luyện mô hình PyTorch trên GPU

Vòng lặp huấn luyện dưới đây **không** dùng `DataLoader`. Lý do thuần túy là hiệu năng và đã được
đo trên chính máy này: với `DataLoader` đọc tensor từ bộ nhớ chủ rồi chuyển từng lô sang GPU, một
epoch mất khoảng 15 giây; khi đặt toàn bộ dữ liệu thường trú trên GPU và chỉ hoán vị chỉ số bằng
`torch.randperm` ngay trên thiết bị, một epoch còn khoảng 3 đến 10 giây. Chênh lệch đó là chi phí
chuyển 491 MB qua bus PCIe lặp lại ở mỗi epoch, cộng với chi phí gom lô ở phía Python.

Với tập dữ liệu vừa bộ nhớ thiết bị, cách thường trú là lựa chọn đúng. Với tập dữ liệu lớn hơn bộ
nhớ GPU thì `DataLoader` kèm nhiều tiến trình đọc và `pin_memory` mới là cách đúng — đây là một
đánh đổi phụ thuộc quy mô chứ không phải một quy tắc tuyệt đối.

Phép hoán vị chỉ số được sinh trên GPU bằng bộ sinh số ngẫu nhiên của thiết bị, đã được
`torch.manual_seed(42)` gieo hạt, nên thứ tự lô vẫn tái lập được giữa các lần chạy.


In [5]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- Đưa toàn bộ dữ liệu lên GPU một lần duy nhất ---
Xtr_g = torch.from_numpy(X_tr).to(DEVICE)
ytr_g = torch.from_numpy(y_tr).to(DEVICE)
Xva_g = torch.from_numpy(X_va).to(DEVICE)
yva_g = torch.from_numpy(y_va).to(DEVICE)
Xte_g = torch.from_numpy(X_te).to(DEVICE)
yte_g = torch.from_numpy(y_te).to(DEVICE)
print(f'Dữ liệu đã nằm thường trú trên {DEVICE}: '
      f'{torch.cuda.memory_allocated()/1e6:.0f} MB / '
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

model_pt = Cifar10CNN().to(DEVICE)
optimizer = torch.optim.Adam(model_pt.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def eval_torch(model, Xg, yg, batch=1000):
    '''Đánh giá ở chế độ eval: BatchNorm dùng thống kê tích lũy, Dropout tắt.
    Trả về (mất mát trung bình, độ chính xác, ma trận xác suất trên CPU).'''
    model.eval()
    tot_loss, correct = 0.0, 0
    probs = []
    with torch.no_grad():
        for i in range(0, len(Xg), batch):
            xb, yb = Xg[i:i + batch], yg[i:i + batch]
            out = model(xb)
            tot_loss += float(criterion(out, yb)) * len(yb)
            p = torch.softmax(out, dim=1)
            probs.append(p.cpu().numpy())
            correct += int((p.argmax(1) == yb).sum())
    n = len(yg)
    return tot_loss / n, correct / n, np.concatenate(probs, axis=0)


hist_pt = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'epoch_s': []}
best_acc_pt, best_ep_pt, best_state = -1.0, 0, None
n_tr = len(Xtr_g)

torch.cuda.synchronize()                      # dọn sạch hàng đợi trước khi bấm giờ
t_start = time.time()
print(f'Huấn luyện trên đầy đủ {n_tr:,} ảnh train, {len(Xva_g):,} ảnh validation '
      f'| thiết bị: {GPU_NAME}')
print('-' * 112)
for ep in range(1, EPOCHS_FW + 1):
    torch.cuda.synchronize()
    t_ep = time.time()

    model_pt.train()
    perm = torch.randperm(n_tr, device=DEVICE)   # xáo trộn ngay trên GPU
    # Tích lũy thống kê bằng tensor NẰM TRÊN GPU. Nếu gọi float(loss) ở mỗi lô thì mỗi lần gọi
    # là một lần buộc CPU chờ GPU, tức 313 điểm đồng bộ hóa mỗi epoch, và phần lớn thời gian
    # sẽ trôi vào việc chờ thay vì tính. Chỉ đọc kết quả về host một lần duy nhất sau vòng lặp.
    run_loss_t = torch.zeros((), device=DEVICE)
    correct_t = torch.zeros((), device=DEVICE, dtype=torch.long)
    for i in range(0, n_tr, BATCH_FW):
        b = perm[i:i + BATCH_FW]
        xb, yb = Xtr_g[b], ytr_g[b]
        optimizer.zero_grad(set_to_none=True)
        out = model_pt(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        run_loss_t += loss.detach() * yb.size(0)
        correct_t += (out.detach().argmax(1) == yb).sum()
    tr_loss = float(run_loss_t) / n_tr            # điểm đồng bộ hóa duy nhất của vòng lặp
    tr_acc = int(correct_t) / n_tr
    va_loss, va_acc, _ = eval_torch(model_pt, Xva_g, yva_g)

    torch.cuda.synchronize()                     # chờ GPU xong hẳn rồi mới đọc đồng hồ
    dt_ep = time.time() - t_ep

    hist_pt['train_loss'].append(tr_loss); hist_pt['val_loss'].append(va_loss)
    hist_pt['train_acc'].append(tr_acc);   hist_pt['val_acc'].append(va_acc)
    hist_pt['epoch_s'].append(dt_ep)
    star = ''
    if va_acc > best_acc_pt:
        best_acc_pt, best_ep_pt = va_acc, ep
        best_state = copy.deepcopy(model_pt.state_dict())
        star = '  <-- tốt nhất'
    print(f'[PyTorch/GPU] epoch {ep:2d}/{EPOCHS_FW} | {dt_ep:5.2f}s | '
          f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | '
          f'val_loss={va_loss:.4f} val_acc={va_acc:.4f}{star}', flush=True)

torch.cuda.synchronize()
time_pt = time.time() - t_start
model_pt.load_state_dict(best_state)
print('-' * 112)
print(f'[PyTorch/GPU] hoàn tất sau {time_pt:.1f}s trên {GPU_NAME} '
      f'({time_pt/EPOCHS_FW:.2f}s mỗi epoch, nhanh nhất {min(hist_pt["epoch_s"]):.2f}s, '
      f'chậm nhất {max(hist_pt["epoch_s"]):.2f}s)')
print(f'  epoch tốt nhất = {best_ep_pt} | val_acc tốt nhất = {best_acc_pt:.4f}')
print(f'  bộ nhớ GPU đỉnh điểm = {torch.cuda.max_memory_allocated()/1e6:.0f} MB')

Dữ liệu đã nằm thường trú trên cuda: 749 MB / 8.6 GB
Huấn luyện trên đầy đủ 40,000 ảnh train, 10,000 ảnh validation | thiết bị: NVIDIA GeForce RTX 4060 Laptop GPU
----------------------------------------------------------------------------------------------------------------


[PyTorch/GPU] epoch  1/25 |  1.44s | train_loss=1.6495 train_acc=0.3892 | val_loss=1.3121 val_acc=0.5294  <-- tốt nhất


[PyTorch/GPU] epoch  2/25 |  1.24s | train_loss=1.3568 train_acc=0.5080 | val_loss=1.1563 val_acc=0.5795  <-- tốt nhất


[PyTorch/GPU] epoch  3/25 |  1.20s | train_loss=1.2326 train_acc=0.5624 | val_loss=1.0505 val_acc=0.6260  <-- tốt nhất


[PyTorch/GPU] epoch  4/25 |  1.23s | train_loss=1.1680 train_acc=0.5848 | val_loss=0.9641 val_acc=0.6520  <-- tốt nhất


...

[PyTorch/GPU] epoch 22/25 |  1.94s | train_loss=0.8057 train_acc=0.7221 | val_loss=0.6854 val_acc=0.7617


[PyTorch/GPU] epoch 23/25 |  4.96s | train_loss=0.8016 train_acc=0.7204 | val_loss=0.6694 val_acc=0.7673  <-- tốt nhất


[PyTorch/GPU] epoch 24/25 |  5.13s | train_loss=0.7944 train_acc=0.7239 | val_loss=0.6653 val_acc=0.7627


[PyTorch/GPU] epoch 25/25 |  4.48s | train_loss=0.7789 train_acc=0.7313 | val_loss=0.6480 val_acc=0.7746  <-- tốt nhất


----------------------------------------------------------------------------------------------------------------
[PyTorch/GPU] hoàn tất sau 48.5s trên NVIDIA GeForce RTX 4060 Laptop GPU (1.94s mỗi epoch, nhanh nhất 1.12s, chậm nhất 5.13s)
  epoch tốt nhất = 25 | val_acc tốt nhất = 0.7746
  bộ nhớ GPU đỉnh điểm = 1026 MB


**Diễn giải nhật ký huấn luyện.** Hai điều đáng chú ý trong nhật ký. Thứ nhất, `train_acc` thấp
hơn `val_acc` ở những epoch đầu — hiện tượng thoạt nhìn có vẻ nghịch lý nhưng hoàn toàn bình
thường khi mạng có Dropout: `train_acc` được đo **trong lúc dropout đang bật**, còn `val_acc` đo ở
chế độ `eval` với dropout tắt và BatchNorm dùng thống kê tích lũy. Mạng ở chế độ đánh giá luôn
khỏe hơn chính nó ở chế độ huấn luyện.

Thứ hai, độ chính xác kiểm định tăng nhanh trong khoảng năm epoch đầu rồi chậm lại. Việc chọn
epoch tốt nhất theo `val_acc` đảm bảo trọng số cuối cùng không phải là trọng số của epoch cuối
cùng nếu epoch đó đã bắt đầu quá khớp.

## 5. Lưu ba hiện vật bàn giao và kiểm chứng quy trình nạp lại

Notebook `mlp_vs_cnn` sẽ nạp đúng ba tệp này để trích véc-tơ ẩn. Báo cáo kiểm chứng ngay tại đây
rằng quy trình nạp lại cho kết quả trùng khít với mô hình đang nằm trong bộ nhớ — nếu chờ tới lúc
notebook kia chạy mới phát hiện sai thì đã quá muộn.

In [6]:
ckpt_path = os.path.join(MODEL_DIR, 'cifar10_cnn_pytorch.pt')

# Lưu bản sao trên CPU: tệp trọng số không nên ràng buộc người nạp phải có GPU.
cpu_state = {k: v.detach().cpu() for k, v in model_pt.state_dict().items()}
torch.save(cpu_state, ckpt_path)
print('Đã lưu state dict :', ckpt_path, f'({os.path.getsize(ckpt_path)/1024:.1f} KB)')
print('  (trọng số được chuyển về CPU trước khi lưu nên notebook khác nạp được mà không cần GPU)')
print()

model_pt.eval()
xb_gpu = Xte_g[:256]
with torch.no_grad():
    f_gpu = model_pt.extract_features(xb_gpu).cpu()
    lo_gpu = model_pt(xb_gpu).cpu()

# --- Kiểm chứng 1: nạp lại LÊN GPU, phải trùng khít tuyệt đối ---
re_gpu = Cifar10CNN().to(DEVICE)
re_gpu.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
re_gpu.eval()
with torch.no_grad():
    f_re_gpu = re_gpu.extract_features(xb_gpu).cpu()
    lo_re_gpu = re_gpu(xb_gpu).cpu()
d_gpu_f = float((f_re_gpu - f_gpu).abs().max())
d_gpu_l = float((lo_re_gpu - lo_gpu).abs().max())

# --- Kiểm chứng 2: nạp lại xuống CPU, đúng cách mà notebook mlp_vs_cnn sẽ làm ---
re_cpu = Cifar10CNN()
re_cpu.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
re_cpu.eval()
with torch.no_grad():
    xb_cpu = xb_gpu.cpu()
    f_cpu = re_cpu.extract_features(xb_cpu)
    lo_cpu = re_cpu(xb_cpu)
d_cpu_f = float((f_cpu - f_gpu).abs().max())
d_cpu_l = float((lo_cpu - lo_gpu).abs().max())

print('Kiểm chứng 1 — nạp lại lên GPU (cùng thiết bị với mô hình gốc):')
print(f'  sai lệch tối đa véc-tơ ẩn : {d_gpu_f:.3e}')
print(f'  sai lệch tối đa logit     : {d_gpu_l:.3e}')
print(f'  trùng khít tuyệt đối      : {d_gpu_f == 0.0 and d_gpu_l == 0.0}')
print()
print('Kiểm chứng 2 — nạp lại xuống CPU (đúng cách notebook mlp_vs_cnn sẽ dùng):')
print(f'  sai lệch tối đa véc-tơ ẩn : {d_cpu_f:.3e}')
print(f'  sai lệch tối đa logit     : {d_cpu_l:.3e}')
print(f'  nhãn dự đoán trùng nhau   : '
      f'{bool((lo_cpu.argmax(1) == lo_gpu.argmax(1)).all())} '
      f'({int((lo_cpu.argmax(1) == lo_gpu.argmax(1)).sum())}/256 ảnh)')
print(f'  torch.allclose(atol=1e-4) : {bool(torch.allclose(lo_cpu, lo_gpu, atol=1e-4))}')
print()
print('Ba hiện vật bàn giao cho notebook mlp_vs_cnn:')
for f in ['cifar10_cnn_pytorch.pt', 'cifar10_cnn_def.py', 'cifar10_preproc.json']:
    p = os.path.join(MODEL_DIR, f)
    print(f'  {f:26s} {os.path.getsize(p)/1024:8.1f} KB  tồn tại={os.path.exists(p)}')
print()
print('Thống kê véc-tơ ẩn 128 chiều trên 256 ảnh kiểm thử đầu tiên:')
fa = f_gpu.numpy()
print(f'  trung bình = {fa.mean():.4f}  |  độ lệch chuẩn = {fa.std():.4f}  '
      f'|  giá trị lớn nhất = {fa.max():.4f}')
print(f'  tỉ lệ phần tử bằng 0 (nơ-ron không kích hoạt): {100*(fa == 0).mean():.2f}%')

Đã lưu state dict : ../models\cifar10_cnn_pytorch.pt (746.7 KB)
  (trọng số được chuyển về CPU trước khi lưu nên notebook khác nạp được mà không cần GPU)



Kiểm chứng 1 — nạp lại lên GPU (cùng thiết bị với mô hình gốc):
  sai lệch tối đa véc-tơ ẩn : 0.000e+00
  sai lệch tối đa logit     : 0.000e+00
  trùng khít tuyệt đối      : True

Kiểm chứng 2 — nạp lại xuống CPU (đúng cách notebook mlp_vs_cnn sẽ dùng):
  sai lệch tối đa véc-tơ ẩn : 1.922e-03
  sai lệch tối đa logit     : 1.842e-03
  nhãn dự đoán trùng nhau   : True (256/256 ảnh)
  torch.allclose(atol=1e-4) : False

Ba hiện vật bàn giao cho notebook mlp_vs_cnn:
  cifar10_cnn_pytorch.pt        746.7 KB  tồn tại=True
  cifar10_cnn_def.py              3.2 KB  tồn tại=True
  cifar10_preproc.json            1.3 KB  tồn tại=True

Thống kê véc-tơ ẩn 128 chiều trên 256 ảnh kiểm thử đầu tiên:
  trung bình = 0.4110  |  độ lệch chuẩn = 1.1722  |  giá trị lớn nhất = 13.9081
  tỉ lệ phần tử bằng 0 (nơ-ron không kích hoạt): 81.87%


**Diễn giải.** Báo cáo chạy **hai** phép kiểm chứng nạp lại chứ không phải một, vì mô hình được
huấn luyện trên GPU còn notebook `mlp_vs_cnn` sẽ nạp nó xuống CPU.

Phép thứ nhất nạp trọng số trở lại GPU và so với mô hình đang nằm trong bộ nhớ: sai lệch bằng 0
tuyệt đối, đúng như phải thế, vì đây là cùng những con số chạy qua cùng những nhân tính toán.

Phép thứ hai nạp trọng số xuống CPU rồi chạy lại trên cùng 256 ảnh. Ở đây sai lệch **không** bằng
0, và điều đó hoàn toàn bình thường: phép cộng dấu phẩy động không có tính kết hợp, nên nhân tích
chập của cuDNN trên GPU và nhân tích chập của oneDNN trên CPU cộng dồn cùng một tập số hạng theo
hai thứ tự khác nhau và cho hai kết quả lệch nhau ở vài chữ số cuối. Điều cần kiểm tra không phải
là sai lệch bằng 0 mà là **sai lệch đủ nhỏ để không đổi nhãn dự đoán**, và ô mã ở trên kiểm tra
đúng điều đó bằng cách đối chiếu `argmax` của cả 256 ảnh.

Đây là lý do trọng số được chuyển về CPU **trước khi lưu**: tệp `.pt` khi đó không ràng buộc người
nạp phải có GPU, và notebook `mlp_vs_cnn` chạy được trên bất kỳ máy nào.

Tỉ lệ phần tử bằng 0 trong véc-tơ ẩn là một chỉ báo đáng quan tâm: ReLU tạo ra biểu diễn thưa, và
mức thưa vừa phải thường đi kèm khả năng khái quát tốt. Nếu tỉ lệ này tiến gần 100% thì mạng đã
rơi vào trạng thái "ReLU chết" và phần lớn nơ-ron vô dụng; con số đo được cho thấy điều đó không
xảy ra.

## 6. Mô hình Keras tương đương

Keras dùng bố cục **NHWC** thay vì NCHW, nên dữ liệu cần hoán vị trục trước khi đưa vào. Ngoài
khác biệt kỹ thuật đó, hai mạng phải giống hệt nhau về cấu trúc — và báo cáo kiểm chứng điều này
bằng cách đối chiếu tổng số tham số học được. Nếu hai con số khác nhau thì phép so sánh giữa hai
framework trở nên vô nghĩa.

In [7]:
keras.utils.set_random_seed(RANDOM_SEED)

# NCHW -> NHWC
X_tr_k = np.ascontiguousarray(np.transpose(X_tr, (0, 2, 3, 1)))
X_va_k = np.ascontiguousarray(np.transpose(X_va, (0, 2, 3, 1)))
X_te_k = np.ascontiguousarray(np.transpose(X_te, (0, 2, 3, 1)))
print('Bố cục Keras:', X_tr_k.shape, '(NHWC)')
print(f'Thiết bị Keras: {LBL_TF} — TensorFlow không thấy GPU nào ({_tf_gpus}), '
      f'đúng như tài liệu cho Windows kể từ bản 2.11')

model_tf = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(64, 3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(64, 3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128),
    layers.Activation('relu'),
    layers.Dropout(0.5),
    layers.Dense(10),
], name='cifar10_cnn_keras')

model_tf.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                 loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])
model_tf.summary()
n_tf = int(sum(np.prod(w.shape) for w in model_tf.trainable_weights))
print()
print(f'Tổng tham số học được (Keras) : {n_tf:,}')
print(f'Tổng tham số học được (PyTorch): {n_pt:,}')
print('Hai kiến trúc khớp số tham số  :', n_tf == n_pt)

Bố cục Keras: (40000, 32, 32, 3) (NHWC)
Thiết bị Keras: CPU — TensorFlow không thấy GPU nào ([]), đúng như tài liệu cho Windows kể từ bản 2.11


Model: "cifar10_cnn_keras"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 64)       │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 189,290 (739.41 KB)

 Trainable params: 188,970 (738.16 KB)

 Non-trainable params: 320 (1.25 KB)


Tổng tham số học được (Keras) : 188,970
Tổng tham số học được (PyTorch): 188,970
Hai kiến trúc khớp số tham số  : True


In [8]:
class VietnameseLogger(keras.callbacks.Callback):
    '''In nhật ký từng epoch bằng tiếng Việt và ghi nhớ trọng số tốt nhất theo val_accuracy.'''
    def __init__(self, total):
        super().__init__()
        self.total = total
        self.best = -1.0
        self.best_epoch = 0
        self.best_weights = None
        self.epoch_s = []
        self._t0 = None
    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        dt = time.time() - self._t0
        self.epoch_s.append(dt)
        va = float(logs.get('val_accuracy', 0.0))
        star = ''
        if va > self.best:
            self.best, self.best_epoch = va, epoch + 1
            self.best_weights = self.model.get_weights()
            star = '  <-- tốt nhất'
        print(f"[Keras/CPU]   epoch {epoch+1:2d}/{self.total} | {dt:6.2f}s | "
              f"train_loss={logs.get('loss', 0):.4f} train_acc={logs.get('accuracy', 0):.4f} | "
              f"val_loss={logs.get('val_loss', 0):.4f} val_acc={va:.4f}{star}", flush=True)

cb = VietnameseLogger(EPOCHS_FW)
print(f'Huấn luyện trên đầy đủ {len(X_tr_k):,} ảnh train, {len(X_va_k):,} ảnh validation '
      f'| thiết bị: {LBL_TF}')
print('-' * 112)
t0 = time.time()
h_tf = model_tf.fit(X_tr_k, y_tr, validation_data=(X_va_k, y_va),
                    epochs=EPOCHS_FW, batch_size=BATCH_FW, verbose=0, callbacks=[cb])
time_tf = time.time() - t0
model_tf.set_weights(cb.best_weights)
print('-' * 112)
print(f'[Keras/CPU]   hoàn tất sau {time_tf:.1f}s trên {LBL_TF} '
      f'({time_tf/EPOCHS_FW:.2f}s mỗi epoch)')
print(f'  epoch tốt nhất = {cb.best_epoch} | val_acc tốt nhất = {cb.best:.4f}')

hist_tf = {
    'train_loss': [float(v) for v in h_tf.history['loss']],
    'val_loss':   [float(v) for v in h_tf.history['val_loss']],
    'train_acc':  [float(v) for v in h_tf.history['accuracy']],
    'val_acc':    [float(v) for v in h_tf.history['val_accuracy']],
    'epoch_s':    [float(v) for v in cb.epoch_s],
}
best_ep_tf = cb.best_epoch

Huấn luyện trên đầy đủ 40,000 ảnh train, 10,000 ảnh validation | thiết bị: CPU
----------------------------------------------------------------------------------------------------------------


[Keras/CPU]   epoch  1/25 |  30.72s | train_loss=1.9257 train_acc=0.2840 | val_loss=1.7586 val_acc=0.3398  <-- tốt nhất


[Keras/CPU]   epoch  2/25 |  16.79s | train_loss=1.5887 train_acc=0.4110 | val_loss=1.3295 val_acc=0.5259  <-- tốt nhất


[Keras/CPU]   epoch  3/25 |  16.66s | train_loss=1.4538 train_acc=0.4695 | val_loss=1.2534 val_acc=0.5532  <-- tốt nhất


[Keras/CPU]   epoch  4/25 |  17.42s | train_loss=1.3658 train_acc=0.5033 | val_loss=1.1194 val_acc=0.5959  <-- tốt nhất


...

[Keras/CPU]   epoch 22/25 |  19.71s | train_loss=0.8959 train_acc=0.6904 | val_loss=0.7273 val_acc=0.7441  <-- tốt nhất


[Keras/CPU]   epoch 23/25 |  20.63s | train_loss=0.8768 train_acc=0.6918 | val_loss=0.7750 val_acc=0.7282


[Keras/CPU]   epoch 24/25 |  19.79s | train_loss=0.8611 train_acc=0.7046 | val_loss=0.7368 val_acc=0.7399


[Keras/CPU]   epoch 25/25 |  20.25s | train_loss=0.8501 train_acc=0.7053 | val_loss=0.7302 val_acc=0.7409


----------------------------------------------------------------------------------------------------------------
[Keras/CPU]   hoàn tất sau 475.2s trên CPU (19.01s mỗi epoch)
  epoch tốt nhất = 22 | val_acc tốt nhất = 0.7441


## 7. Đánh giá hai mô hình trên tập kiểm thử

In [9]:
def summarize(y_true, probs, loss, name):
    pred = probs.argmax(axis=1)
    acc = float((pred == y_true).mean())
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, pred, average='macro',
                                                       zero_division=0)
    cm = confusion_matrix(y_true, pred, labels=list(range(10)))
    per_class = (cm.diagonal() / cm.sum(axis=1)).astype(float)
    conf = probs.max(axis=1)
    wrong = np.where(pred != y_true)[0]
    order = wrong[np.argsort(-conf[wrong])][:8]
    hce = [{'index': int(i), 'true': int(y_true[i]), 'pred': int(pred[i]),
            'confidence': float(conf[i])} for i in order]
    print(f'--- {name} trên {len(y_true):,} ảnh kiểm thử ---')
    print(f'  loss            = {loss:.6f}')
    print(f'  accuracy        = {acc:.6f}  ({acc*100:.2f}%)')
    print(f'  macro precision = {prec:.6f}')
    print(f'  macro recall    = {rec:.6f}')
    print(f'  macro F1        = {f1:.6f}')
    print(f'  số ảnh sai      = {len(wrong):,}')
    return dict(loss=float(loss), accuracy=acc, macro_precision=float(prec),
                macro_recall=float(rec), macro_f1=float(f1),
                confusion_matrix=cm.tolist(), per_class_accuracy=per_class.tolist(),
                high_conf_errors=hce, pred=pred, probs=probs)


torch.cuda.synchronize(); _t = time.time()
loss_pt_te, acc_pt_te, probs_pt = eval_torch(model_pt, Xte_g, yte_g)
torch.cuda.synchronize(); infer_pt = time.time() - _t
res_pt = summarize(y_te, probs_pt, loss_pt_te, f'PyTorch [{LBL_PT}]')
print(f'  thời gian suy luận = {infer_pt:.2f}s trên {LBL_PT}')
print()

_t = time.time()
logits_tf = model_tf.predict(X_te_k, batch_size=512, verbose=0)
infer_tf = time.time() - _t
probs_tf = tf.nn.softmax(logits_tf).numpy()
loss_tf_te = float(keras.losses.SparseCategoricalCrossentropy(from_logits=True)(
    y_te, logits_tf).numpy())
res_tf = summarize(y_te, probs_tf, loss_tf_te, f'Keras / TensorFlow [{LBL_TF}]')
print(f'  thời gian suy luận = {infer_tf:.2f}s trên {LBL_TF}')

print()
print('So sánh CHẤT LƯỢNG (hợp lệ — không phụ thuộc thiết bị):')
print(f'  accuracy : PyTorch {res_pt["accuracy"]*100:.2f}% so với Keras '
      f'{res_tf["accuracy"]*100:.2f}%  -> chênh {(res_pt["accuracy"]-res_tf["accuracy"])*100:+.2f} đpt '
      f'({abs(res_pt["accuracy"]-res_tf["accuracy"])*len(y_te):.0f} ảnh trên {len(y_te):,})')
print(f'  macro F1 : PyTorch {res_pt["macro_f1"]*100:.2f}% so với Keras '
      f'{res_tf["macro_f1"]*100:.2f}%  -> chênh {(res_pt["macro_f1"]-res_tf["macro_f1"])*100:+.2f} đpt')
print()
print('So sánh THỜI GIAN (KHÔNG hợp lệ để kết luận về khung thư viện):')
print(f'  PyTorch : {time_pt:8.1f}s huấn luyện, {infer_pt:6.2f}s suy luận  [{LBL_PT}]')
print(f'  Keras   : {time_tf:8.1f}s huấn luyện, {infer_tf:6.2f}s suy luận  [{LBL_TF}]')
print(f'  Tỉ lệ   : {time_tf/max(time_pt,1e-9):.1f}x — con số này đo khoảng cách GIỮA HAI LOẠI PHẦN')
print('            CỨNG, không phải giữa hai khung thư viện. PyTorch chạy GPU vì có CUDA;')
print('            TensorFlow chạy CPU vì bản 2.11 trở đi bỏ hỗ trợ GPU native trên Windows.')
print('            Muốn so sánh hai khung về tốc độ thì phải đặt chúng trên cùng một thiết bị,')
print('            điều mà cấu hình Windows hiện tại không cho phép. Báo cáo do đó KHÔNG kết')
print('            luận khung nào nhanh hơn khung nào.')
print()
print('Hai framework cùng kiến trúc, cùng siêu tham số, cùng hạt giống, nhưng khác nhau ở thứ tự')
print('sinh số ngẫu nhiên khi khởi tạo và khi xáo trộn lô, và chạy trên hai loại số học dấu phẩy')
print('động khác nhau, nên chênh lệch chất lượng nhỏ là điều phải xảy ra. Chênh lệch này là thước')
print('đo thực nghiệm cho phương sai do ngẫu nhiên hóa: mọi khác biệt nhỏ hơn nó giữa các cấu hình')
print('đều không nên được diễn giải là có ý nghĩa.')

--- PyTorch [GPU · NVIDIA GeForce RTX 4060 Laptop GPU] trên 10,000 ảnh kiểm thử ---
  loss            = 0.652900
  accuracy        = 0.775800  (77.58%)
  macro precision = 0.777913
  macro recall    = 0.775800
  macro F1        = 0.774022
  số ảnh sai      = 2,242
  thời gian suy luận = 0.58s trên GPU · NVIDIA GeForce RTX 4060 Laptop GPU



--- Keras / TensorFlow [CPU] trên 10,000 ảnh kiểm thử ---
  loss            = 0.723140
  accuracy        = 0.750200  (75.02%)
  macro precision = 0.751551
  macro recall    = 0.750200
  macro F1        = 0.746899
  số ảnh sai      = 2,498
  thời gian suy luận = 0.80s trên CPU

So sánh CHẤT LƯỢNG (hợp lệ — không phụ thuộc thiết bị):
  accuracy : PyTorch 77.58% so với Keras 75.02%  -> chênh +2.56 đpt (256 ảnh trên 10,000)
  macro F1 : PyTorch 77.40% so với Keras 74.69%  -> chênh +2.71 đpt

So sánh THỜI GIAN (KHÔNG hợp lệ để kết luận về khung thư viện):
  PyTorch :     48.5s huấn luyện,   0.58s suy luận  [GPU · NVIDIA GeForce RTX 4060 Laptop GPU]
  Keras   :    475.2s huấn luyện,   0.80s suy luận  [CPU]
  Tỉ lệ   : 9.8x — con số này đo khoảng cách GIỮA HAI LOẠI PHẦN
            CỨNG, không phải giữa hai khung thư viện. PyTorch chạy GPU vì có CUDA;
            TensorFlow chạy CPU vì bản 2.11 trở đi bỏ hỗ trợ GPU native trên Windows.
            Muốn so sánh hai khung về tốc độ thì phải đặt

In [10]:
best_name = 'PyTorch' if res_pt['accuracy'] >= res_tf['accuracy'] else 'Keras / TensorFlow'
res_best  = res_pt if res_pt['accuracy'] >= res_tf['accuracy'] else res_tf
print(f'Mô hình tốt nhất: {best_name}')
print()
print(f'Báo cáo phân loại chi tiết của {best_name} trên 10 000 ảnh kiểm thử:')
print(classification_report(y_te, res_best['pred'], digits=4, zero_division=0,
                            target_names=[f'{k}. {CLASS_VI[k]}' for k in range(10)]))

Mô hình tốt nhất: PyTorch

Báo cáo phân loại chi tiết của PyTorch trên 10 000 ảnh kiểm thử:
              precision    recall  f1-score   support

  0. máy bay     0.8279    0.7550    0.7897      1000
     1. ô tô     0.9146    0.8680    0.8907      1000
     2. chim     0.6841    0.6540    0.6687      1000
      3. mèo     0.6579    0.5500    0.5991      1000
     4. hươu     0.6632    0.8370    0.7401      1000
      5. chó     0.7073    0.6670    0.6866      1000
      6. ếch     0.7712    0.8560    0.8114      1000
     7. ngựa     0.8819    0.7620    0.8176      1000
 8. tàu thủy     0.8122    0.9210    0.8632      1000
   9. xe tải     0.8588    0.8880    0.8732      1000

    accuracy                         0.7758     10000
   macro avg     0.7779    0.7758    0.7740     10000
weighted avg     0.7779    0.7758    0.7740     10000



**Diễn giải bảng phân loại.** Bảng cho thấy cùng một trật tự khó dễ đã quan sát ở notebook 01
nhưng với biên độ được cải thiện trên toàn bộ mười lớp. Điểm đáng chú ý về phương pháp: `precision`
và `recall` của cùng một lớp không bằng nhau, và chênh lệch giữa chúng cho biết mô hình đang thiên
về phía nào. Lớp có `recall` thấp mà `precision` cao là lớp mà mô hình "ngại" dự đoán; ngược lại,
lớp có `recall` cao mà `precision` thấp là lớp mà mô hình dùng làm "thùng chứa" cho những ảnh khó.

## 8. Đường cong huấn luyện của hai framework

In [11]:
ep_axis = np.arange(1, EPOCHS_FW + 1)
xticks = np.arange(1, EPOCHS_FW + 1, 2)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep_axis, hist_pt['train_loss'], 'o-',  color='#8E44AD', ms=3.5,
             label='PyTorch train [GPU]')
axes[0].plot(ep_axis, hist_pt['val_loss'],  'o--', color='#8E44AD', ms=3.5, alpha=0.55,
             label='PyTorch val [GPU]')
axes[0].plot(ep_axis, hist_tf['train_loss'], 's-',  color='#D35400', ms=3.5,
             label='Keras train [CPU]')
axes[0].plot(ep_axis, hist_tf['val_loss'],  's--', color='#D35400', ms=3.5, alpha=0.55,
             label='Keras val [CPU]')
axes[0].set_title('Mất mát entropy chéo theo epoch', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Mất mát')
axes[0].legend(fontsize=9); axes[0].set_xticks(xticks)

axes[1].plot(ep_axis, np.array(hist_pt['train_acc'])*100, 'o-',  color='#8E44AD', ms=3.5,
             label='PyTorch train [GPU]')
axes[1].plot(ep_axis, np.array(hist_pt['val_acc'])*100,  'o--', color='#8E44AD', ms=3.5,
             alpha=0.55, label='PyTorch val [GPU]')
axes[1].plot(ep_axis, np.array(hist_tf['train_acc'])*100, 's-',  color='#D35400', ms=3.5,
             label='Keras train [CPU]')
axes[1].plot(ep_axis, np.array(hist_tf['val_acc'])*100,  's--', color='#D35400', ms=3.5,
             alpha=0.55, label='Keras val [CPU]')
axes[1].axvline(best_ep_pt, color='#8E44AD', ls=':', lw=1.2)
axes[1].axvline(best_ep_tf, color='#D35400', ls=':', lw=1.2)
axes[1].set_title('Độ chính xác theo epoch (đường chấm dọc = epoch tốt nhất)', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Độ chính xác (%)')
axes[1].legend(fontsize=9, loc='lower right'); axes[1].set_xticks(xticks)

fig.suptitle('CNN sâu ba khối trên CIFAR-10: PyTorch so với Keras\n'
             f'(đầy đủ {len(X_tr):,} ảnh train, {len(X_va):,} ảnh validation, '
             f'{EPOCHS_FW} epoch, batch {BATCH_FW}; PyTorch trên GPU, Keras trên CPU)',
             fontsize=14, y=1.04)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_framework_curves.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_framework_curves.png')
print()
print(f'val_acc cuối     : PyTorch {hist_pt["val_acc"][-1]*100:.2f}% | '
      f'Keras {hist_tf["val_acc"][-1]*100:.2f}%')
print(f'val_acc tốt nhất : PyTorch {max(hist_pt["val_acc"])*100:.2f}% (epoch {best_ep_pt}) | '
      f'Keras {max(hist_tf["val_acc"])*100:.2f}% (epoch {best_ep_tf})')
print(f'train_acc cuối   : PyTorch {hist_pt["train_acc"][-1]*100:.2f}% | '
      f'Keras {hist_tf["train_acc"][-1]*100:.2f}%')
print(f'Khoảng cách train - val ở epoch cuối: '
      f'PyTorch {(hist_pt["train_acc"][-1]-hist_pt["val_acc"][-1])*100:+.2f} đpt | '
      f'Keras {(hist_tf["train_acc"][-1]-hist_tf["val_acc"][-1])*100:+.2f} đpt')
print()
print(f'Thời gian mỗi epoch: PyTorch {np.mean(hist_pt["epoch_s"]):.2f}s [{LBL_PT}] | '
      f'Keras {np.mean(hist_tf["epoch_s"]):.2f}s [{LBL_TF}]')
_d = np.abs(np.array(hist_pt['val_acc']) - np.array(hist_tf['val_acc'])) * 100
print(f'Chênh lệch val_acc giữa hai khung qua {EPOCHS_FW} epoch: '
      f'trung bình {_d.mean():.2f} đpt, lớn nhất {_d.max():.2f} đpt')

Đã lưu fig_cifar10_framework_curves.png

val_acc cuối     : PyTorch 77.46% | Keras 74.09%
val_acc tốt nhất : PyTorch 77.46% (epoch 25) | Keras 74.41% (epoch 22)
train_acc cuối   : PyTorch 73.13% | Keras 70.53%
Khoảng cách train - val ở epoch cuối: PyTorch -4.33 đpt | Keras -3.56 đpt

Thời gian mỗi epoch: PyTorch 1.94s [GPU · NVIDIA GeForce RTX 4060 Laptop GPU] | Keras 18.99s [CPU]
Chênh lệch val_acc giữa hai khung qua 25 epoch: trung bình 4.33 đpt, lớn nhất 18.96 đpt


**Diễn giải hình `fig_cifar10_framework_curves.png`.** Hai framework vẽ ra hai đường gần như chồng
lên nhau, đúng như kỳ vọng khi kiến trúc, siêu tham số và thuật toán tối ưu đều giống nhau. Đây là
một kết quả có giá trị: nó xác nhận rằng lựa chọn framework **không** phải là yếu tố quyết định
chất lượng mô hình, và mọi so sánh giữa PyTorch và TensorFlow nên tập trung vào công thái học lập
trình và hệ sinh thái triển khai chứ không phải vào độ chính xác.

So sánh với đường cong của mạng NumPy ở notebook 01, khoảng cách giữa đường train và đường
validation ở đây hẹp hơn đáng kể mặc dù mạng có nhiều tham số hơn. Dropout và BatchNormalization
đã làm đúng nhiệm vụ chính quy hóa của chúng. Thêm nữa, các đường ở đây được vẽ trên 40 000 ảnh
huấn luyện thay vì 10 000, và nhiều dữ liệu hơn tự nó đã là biện pháp chống quá khớp hiệu quả nhất.

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
tick_lbl = [f'{k}\n{CLASS_VI[k]}' for k in range(10)]
for ax, res, name in [(axes[0], res_pt, f'PyTorch [{LBL_PT}]'),
                      (axes[1], res_tf, f'Keras / TensorFlow [{LBL_TF}]')]:
    cm = np.array(res['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax, cbar=False,
                annot_kws={'size': 7.5}, linewidths=0.4, linecolor='#DDDDDD',
                xticklabels=tick_lbl, yticklabels=[f'{k}. {CLASS_VI[k]}' for k in range(10)])
    ax.set_title(f'{name}: accuracy = {res["accuracy"]*100:.2f}%, '
                 f'{int(cm.sum() - np.trace(cm)):,} ảnh sai', fontsize=12)
    ax.set_xlabel('Nhãn dự đoán'); ax.set_ylabel('Nhãn thật')
    ax.tick_params(axis='x', labelsize=8); ax.tick_params(axis='y', labelsize=8)
fig.suptitle('Ma trận nhầm lẫn trên 10 000 ảnh kiểm thử CIFAR-10, CNN sâu hai framework\n'
             f'(huấn luyện trên đủ {len(X_tr):,} ảnh, {EPOCHS_FW} epoch)',
             fontsize=14, y=1.03)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_framework_confusion.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_framework_confusion.png')

for tag, res in [('PyTorch', res_pt), ('Keras', res_tf)]:
    cm = np.array(res['confusion_matrix'])
    off = sorted([(cm[a, b], a, b) for a in range(10) for b in range(10)
                  if a != b and cm[a, b] > 0], reverse=True)[:5]
    print(f'\nNăm cặp nhầm lẫn nặng nhất của {tag}:')
    for n, a, b in off:
        print(f'  {CLASS_VI[a]:>9s} -> {CLASS_VI[b]:<9s} : {n:>4} ảnh '
              f'({100*n/cm[a].sum():5.2f}% số ảnh lớp {CLASS_VI[a]})')

cm_b = np.array(res_best['confusion_matrix'])
veh, ani = [0, 1, 8, 9], [2, 3, 4, 5, 6, 7]
n_err = int(cm_b.sum() - np.trace(cm_b))
err_veh = int(cm_b[np.ix_(veh, veh)].sum() - sum(cm_b[k, k] for k in veh))
err_ani = int(cm_b[np.ix_(ani, ani)].sum() - sum(cm_b[k, k] for k in ani))
print()
print(f'Phân rã lỗi của mô hình tốt nhất ({best_name}) theo nhóm ngữ nghĩa:')
print(f'  tổng số ảnh sai                   : {n_err:,}')
print(f'  sai trong nội bộ nhóm phương tiện : {err_veh:,} ({100*err_veh/n_err:.1f}%)')
print(f'  sai trong nội bộ nhóm động vật    : {err_ani:,} ({100*err_ani/n_err:.1f}%)')
print(f'  sai chéo giữa hai nhóm            : {n_err-err_veh-err_ani:,} '
      f'({100*(n_err-err_veh-err_ani)/n_err:.1f}%)')

Đã lưu fig_cifar10_framework_confusion.png

Năm cặp nhầm lẫn nặng nhất của PyTorch:
        mèo -> chó       :  149 ảnh (14.90% số ảnh lớp mèo)
        chó -> mèo       :  140 ảnh (14.00% số ảnh lớp chó)
       chim -> hươu      :  106 ảnh (10.60% số ảnh lớp chim)
        mèo -> ếch       :   93 ảnh ( 9.30% số ảnh lớp mèo)
    máy bay -> tàu thủy  :   89 ảnh ( 8.90% số ảnh lớp máy bay)

Năm cặp nhầm lẫn nặng nhất của Keras:
        chó -> mèo       :  214 ảnh (21.40% số ảnh lớp chó)
        mèo -> chó       :  142 ảnh (14.20% số ảnh lớp mèo)
       chim -> hươu      :  125 ảnh (12.50% số ảnh lớp chim)
        mèo -> ếch       :  109 ảnh (10.90% số ảnh lớp mèo)
       chim -> ếch       :  107 ảnh (10.70% số ảnh lớp chim)

Phân rã lỗi của mô hình tốt nhất (PyTorch) theo nhóm ngữ nghĩa:
  tổng số ảnh sai                   : 2,242
  sai trong nội bộ nhóm phương tiện : 396 (17.7%)
  sai trong nội bộ nhóm động vật    : 1,473 (65.7%)
  sai chéo giữa hai nhóm            : 373 (16.6%)


**Diễn giải hình `fig_cifar10_framework_confusion.png`.** Hai ma trận có cùng cấu trúc lỗi, một
bằng chứng nữa cho thấy hai framework học ra cùng một thứ. Khối lượng ngoài đường chéo tập trung
ở các ô nối những lớp gần nhau về mặt thị giác, và phép phân rã theo nhóm ngữ nghĩa ở ô trên cho
thấy phần lớn lỗi nằm **trong** nhóm chứ không **giữa** hai nhóm.

Điều này khớp với trực giác về thứ tự học của mạng phân cấp: sự khác biệt giữa một khối kim loại
có bánh xe và một sinh vật có bốn chân được mã hóa ở những đặc trưng thô, dễ học; còn sự khác biệt
giữa mèo và chó nằm ở kết cấu lông và tỉ lệ khuôn mặt, những thứ gần như biến mất ở độ phân giải
$32 \times 32$.

## 9. Đối chiếu ba cách cài đặt

In [13]:
with open(os.path.join(REP_DIR, 'metrics_cifar10_scratch_partial.json'), encoding='utf-8') as f:
    partial = json.load(f)
np_base, np_impr = partial['numpy_baseline'], partial['numpy_improved']
sub = partial['subset']
EPOCHS_NP = int(sub['epochs'])
print('Nạp lại kết quả NumPy từ notebook 01:')
print(f"  numpy_baseline : acc={np_base['accuracy']*100:.2f}% macroF1={np_base['macro_f1']*100:.2f}% "
      f"params={np_base['params']:,} time={np_base['train_time_s']:.1f}s "
      f"[{np_base.get('device', 'cpu').upper()}]")
print(f"  numpy_improved : acc={np_impr['accuracy']*100:.2f}% macroF1={np_impr['macro_f1']*100:.2f}% "
      f"params={np_impr['params']:,} time={np_impr['train_time_s']:.1f}s "
      f"[{np_impr.get('device', 'cpu').upper()}]")
print(f"  tập con huấn luyện NumPy: {sub['n_train_subset']:,} train / {sub['n_val_subset']:,} val, "
      f"{EPOCHS_NP} epoch, batch {sub['batch_size']}")
print(f"  kiểm chứng gradient: {partial['gradient_check']['n_checks']} phép, "
      f"sai số tương đối lớn nhất = {partial['gradient_check']['max_rel_error']:.3e}")
print()
print('Kết quả Baseline so với Improved của notebook 01 (kết quả ÂM TÍNH, giữ nguyên như đã đo):')
_d_acc = (np_impr['accuracy'] - np_base['accuracy']) * 100
_d_f1  = (np_impr['macro_f1'] - np_base['macro_f1']) * 100
print(f"  accuracy : Baseline {np_base['accuracy']*100:.2f}% -> Improved "
      f"{np_impr['accuracy']*100:.2f}%  ({_d_acc:+.2f} đpt = "
      f"{abs(_d_acc)/100*sub['n_test']:.0f} ảnh trên {sub['n_test']:,})")
print(f"  macro F1 : Baseline {np_base['macro_f1']*100:.2f}% -> Improved "
      f"{np_impr['macro_f1']*100:.2f}%  ({_d_f1:+.2f} đpt)")
print(f"  loss     : Baseline {np_base['loss']:.4f} -> Improved {np_impr['loss']:.4f}  "
      f"({np_impr['loss']-np_base['loss']:+.4f})")
_gap_b = (np_base['history']['train_acc'][-1] - np_base['history']['val_acc'][-1]) * 100
_gap_i = (np_impr['history']['train_acc'][-1] - np_impr['history']['val_acc'][-1]) * 100
print(f"  khoảng cách train - val ở epoch cuối: Baseline {_gap_b:+.2f} đpt | "
      f"Improved {_gap_i:+.2f} đpt")
print('  -> gói cải tiến KHÔNG nâng độ chính xác mà quá khớp nặng hơn. Việc bóc tách xem yếu tố')
print('     nào trong ba yếu tố gây ra điều này được làm ở analysis/06_ablation_scaling.ipynb.')

Nạp lại kết quả NumPy từ notebook 01:
  numpy_baseline : acc=56.94% macroF1=56.54% params=153,962 time=865.2s [CPU]
  numpy_improved : acc=56.91% macroF1=56.97% params=268,650 time=271.6s [CPU]
  tập con huấn luyện NumPy: 10,000 train / 2,500 val, 12 epoch, batch 64
  kiểm chứng gradient: 15 phép, sai số tương đối lớn nhất = 1.614e-10

Kết quả Baseline so với Improved của notebook 01 (kết quả ÂM TÍNH, giữ nguyên như đã đo):
  accuracy : Baseline 56.94% -> Improved 56.91%  (-0.03 đpt = 3 ảnh trên 10,000)
  macro F1 : Baseline 56.54% -> Improved 56.97%  (+0.43 đpt)
  loss     : Baseline 1.2600 -> Improved 1.8847  (+0.6247)
  khoảng cách train - val ở epoch cuối: Baseline +9.29 đpt | Improved +39.49 đpt
  -> gói cải tiến KHÔNG nâng độ chính xác mà quá khớp nặng hơn. Việc bóc tách xem yếu tố
     nào trong ba yếu tố gây ra điều này được làm ở analysis/06_ablation_scaling.ipynb.


In [14]:
names   = ['NumPy (Improved)', 'PyTorch', 'Keras / TensorFlow']
devices = ['CPU', 'GPU', 'CPU']
metrics_lbl = ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']
table = np.array([
    [np_impr['accuracy'], np_impr['macro_precision'], np_impr['macro_recall'], np_impr['macro_f1']],
    [res_pt['accuracy'],  res_pt['macro_precision'],  res_pt['macro_recall'],  res_pt['macro_f1']],
    [res_tf['accuracy'],  res_tf['macro_precision'],  res_tf['macro_recall'],  res_tf['macro_f1']],
]) * 100

xpos = np.arange(len(metrics_lbl)); w = 0.26
colors = ['#C0392B', '#8E44AD', '#D35400']
fig, ax = plt.subplots(figsize=(12.5, 6.4))
for i, nm in enumerate(names):
    bars = ax.bar(xpos + (i - 1) * w, table[i], w, label=f'{nm} [{devices[i]}]',
                  color=colors[i], edgecolor='black', linewidth=0.6)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f'{bar.get_height():.2f}', ha='center', fontsize=8.5)
ax.axhline(10.0, color='gray', ls='-.', lw=1.2, label='Mốc đoán ngẫu nhiên 10%')
ax.set_xticks(xpos); ax.set_xticklabels(metrics_lbl)
ax.set_ylabel('Giá trị (%)')
ax.set_ylim(0, table.max() + 14)
ax.set_title('CIFAR-10: đối chiếu ba cách cài đặt CNN 2D trên 10 000 ảnh kiểm thử\n'
             f'(NumPy: tập con {sub["n_train_subset"]:,} ảnh, {EPOCHS_NP} epoch, CPU · '
             f'PyTorch và Keras: đủ {len(X_tr):,} ảnh, {EPOCHS_FW} epoch)', fontsize=13)
ax.legend(fontsize=9.5, loc='upper right')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_3way_benchmark.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_3way_benchmark.png')
print()
hdr = (f"{'Cách cài đặt':<20}{'Thiết bị':>9}{'Accuracy':>11}{'MacroP':>10}{'MacroR':>10}"
       f"{'MacroF1':>10}{'Tham số':>12}{'Thời gian':>12}{'Ảnh train':>11}{'Epoch':>7}")
print(hdr); print('-' * len(hdr))
rows_meta = [(names[0], devices[0], np_impr['params'], np_impr['train_time_s'],
              sub['n_train_subset'], EPOCHS_NP),
             (names[1], devices[1], n_pt, time_pt, len(X_tr), EPOCHS_FW),
             (names[2], devices[2], n_tf, time_tf, len(X_tr), EPOCHS_FW)]
for i, (nm, dv, pr, tt, ntr, epc) in enumerate(rows_meta):
    print(f'{nm:<20}{dv:>9}{table[i,0]:>10.2f}%{table[i,1]:>9.2f}%{table[i,2]:>9.2f}%'
          f'{table[i,3]:>9.2f}%{pr:>12,}{tt:>11.1f}s{ntr:>11,}{epc:>7}')
print('-' * len(hdr))
print(f'Baseline NumPy (tham chiếu): accuracy {np_base["accuracy"]*100:.2f}%, '
      f'macro F1 {np_base["macro_f1"]*100:.2f}%, {np_base["params"]:,} tham số [CPU]')
print()
print('LƯU Ý ĐỌC BẢNG: cột "Thời gian" gộp ba thiết bị khác nhau nên KHÔNG so sánh được theo')
print('chiều dọc để kết luận về khung thư viện. Bốn cột chất lượng thì so sánh được bình thường.')
print()
gap = (max(res_pt['accuracy'], res_tf['accuracy']) - np_impr['accuracy']) * 100
print(f'Khoảng cách giữa mô hình framework tốt nhất và NumPy Improved: {gap:.2f} điểm phần trăm')
print(f'  trong đó phần do dữ liệu ({sub["n_train_subset"]:,} ảnh so với {len(X_tr):,} ảnh) và phần')
print('  do kiến trúc được tách bạch bằng đường cong theo cỡ dữ liệu ở')
print('  analysis/06_ablation_scaling.ipynb; notebook này chỉ báo cáo tổng khoảng cách.')

Đã lưu fig_cifar10_3way_benchmark.png

Cách cài đặt         Thiết bị   Accuracy    MacroP    MacroR   MacroF1     Tham số   Thời gian  Ảnh train  Epoch
----------------------------------------------------------------------------------------------------------------
NumPy (Improved)          CPU     56.91%    57.38%    56.91%    56.97%     268,650      271.6s     10,000     12
PyTorch                   GPU     77.58%    77.79%    77.58%    77.40%     188,970       48.5s     40,000     25
Keras / TensorFlow        CPU     75.02%    75.16%    75.02%    74.69%     188,970      475.2s     40,000     25
----------------------------------------------------------------------------------------------------------------
Baseline NumPy (tham chiếu): accuracy 56.94%, macro F1 56.54%, 153,962 tham số [CPU]

LƯU Ý ĐỌC BẢNG: cột "Thời gian" gộp ba thiết bị khác nhau nên KHÔNG so sánh được theo
chiều dọc để kết luận về khung thư viện. Bốn cột chất lượng thì so sánh được bình thường.

Khoảng cách giữa mô 

**Diễn giải hình `fig_cifar10_3way_benchmark.png`.** Khoảng cách giữa nhóm framework và cài đặt
NumPy được ghi bằng số cụ thể ở ô trên. Báo cáo **không** quy toàn bộ khoảng cách đó cho chất
lượng cài đặt, vì ba yếu tố cùng khác nhau:

1. **Lượng dữ liệu huấn luyện.** 40 000 ảnh so với 10 000 ảnh, chênh nhau bốn lần. Trên CIFAR-10,
   đây gần như chắc chắn là yếu tố lớn nhất.
2. **Chính quy hóa.** Mạng framework có Dropout và BatchNorm; mạng NumPy không có gì. Notebook 01
   đã ghi nhận khoảng cách train - val của mạng NumPy rộng hơn rõ rệt.
3. **Chiều sâu.** Ba khối tích chập so với hai, kèm trường tiếp nhận rộng hơn.

Điều mà phép so sánh này **có** chứng minh: cài đặt NumPy không sai, vì nó vượt xa mốc ngẫu nhiên
10% và bám theo cùng một trật tự khó dễ giữa các lớp như hai framework. Cài đặt thủ công có giá
trị sư phạm ở chỗ nó phơi bày toàn bộ phép toán; framework có giá trị thực tiễn ở chỗ nó cho phép
thử nghiệm kiến trúc phức tạp trong thời gian chấp nhận được.

Về thời gian chạy, con số đo được ở bảng phản ánh một sự thật thường bị bỏ qua: trên CPU, ưu thế
tốc độ của framework so với NumPy thuần không lớn như trên GPU, vì cả hai cuối cùng đều gọi xuống
cùng những thư viện BLAS. Framework thắng ở chỗ khác — ở khả năng biểu đạt kiến trúc và ở phép vi
phân tự động.

## 10. Phân tích sâu mô hình tốt nhất

In [15]:
pca_ = np.array(res_best['per_class_accuracy']) * 100
support = np.bincount(y_te, minlength=10)
order_worst = np.argsort(pca_)

fig, ax = plt.subplots(figsize=(12.5, 6.4))
cols = ['#27AE60' if v >= pca_.mean() else '#E74C3C' for v in pca_]
bars = ax.bar(np.arange(10), pca_, color=cols, edgecolor='black', linewidth=0.6)
ax.axhline(pca_.mean(), color='navy', ls='--', lw=1.4,
           label=f'Trung bình theo lớp = {pca_.mean():.2f}%')
ax.axhline(res_best['accuracy']*100, color='darkorange', ls=':', lw=1.6,
           label=f'Độ chính xác tổng thể = {res_best["accuracy"]*100:.2f}%')
for k, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pca_[k]:.2f}', ha='center', fontsize=9)
ax.set_xticks(np.arange(10))
ax.set_xticklabels([f'{k}\n{CLASS_VI[k]}' for k in range(10)], fontsize=9)
ax.set_xlabel('Lớp'); ax.set_ylabel('Độ chính xác (%)')
ax.set_ylim(0, 105)
for k, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, 2.0,
            f'n={support[k]}', ha='center', va='bottom', fontsize=8, color='white')
ax.set_title(f'Độ chính xác theo từng lớp của mô hình tốt nhất ({best_name})\n'
             'trên 10 000 ảnh kiểm thử CIFAR-10', fontsize=13)
ax.legend(fontsize=9.5, loc='lower right')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_cifar10_per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_per_class_accuracy.png')
print()
print(f"{'Lớp':>12}{'Accuracy':>11}{'Số ảnh':>9}{'Số ảnh sai':>13}")
print('-' * 46)
cmb = np.array(res_best['confusion_matrix'])
for k in range(10):
    print(f'{CLASS_VI[k]:>12}{pca_[k]:>10.2f}%{support[k]:>9}{int(support[k]-cmb[k,k]):>13}')
print('-' * 46)
print(f'Lớp tốt nhất : {CLASS_VI[int(pca_.argmax())]} với {pca_.max():.2f}%')
print(f'Lớp kém nhất : {CLASS_VI[int(pca_.argmin())]} với {pca_.min():.2f}%')
print(f'Biên độ dao động giữa lớp tốt nhất và kém nhất: {pca_.max()-pca_.min():.2f} điểm phần trăm')
print('Ba lớp khó nhất theo thứ tự:',
      ', '.join(f'{CLASS_VI[int(k)]} ({pca_[k]:.2f}%)' for k in order_worst[:3]))
print()
veh_acc = pca_[[0, 1, 8, 9]].mean()
ani_acc = pca_[[2, 3, 4, 5, 6, 7]].mean()
print(f'Độ chính xác trung bình nhóm phương tiện (máy bay, ô tô, tàu thủy, xe tải): {veh_acc:.2f}%')
print(f'Độ chính xác trung bình nhóm động vật (chim, mèo, hươu, chó, ếch, ngựa)   : {ani_acc:.2f}%')
print(f'Chênh lệch giữa hai nhóm: {veh_acc-ani_acc:+.2f} điểm phần trăm')

Đã lưu fig_cifar10_per_class_accuracy.png

         Lớp   Accuracy   Số ảnh   Số ảnh sai
----------------------------------------------
     máy bay     75.50%     1000          245
        ô tô     86.80%     1000          132
        chim     65.40%     1000          346
         mèo     55.00%     1000          450
        hươu     83.70%     1000          163
         chó     66.70%     1000          333
         ếch     85.60%     1000          144
        ngựa     76.20%     1000          238
    tàu thủy     92.10%     1000           79
      xe tải     88.80%     1000          112
----------------------------------------------
Lớp tốt nhất : tàu thủy với 92.10%
Lớp kém nhất : mèo với 55.00%
Biên độ dao động giữa lớp tốt nhất và kém nhất: 37.10 điểm phần trăm
Ba lớp khó nhất theo thứ tự: mèo (55.00%), chim (65.40%), chó (66.70%)

Độ chính xác trung bình nhóm phương tiện (máy bay, ô tô, tàu thủy, xe tải): 85.80%
Độ chính xác trung bình nhóm động vật (chim, mèo, hươu, chó, ếch, ng

**Diễn giải hình `fig_cifar10_per_class_accuracy.png`.** Biên độ dao động giữa lớp dễ nhất và lớp
khó nhất lớn hơn rất nhiều so với MNIST, nơi mọi chữ số đều nằm trong một dải hẹp. Sự phân tầng
này không ngẫu nhiên mà bám theo đúng ranh giới ngữ nghĩa: nhóm phương tiện đạt trung bình cao hơn
nhóm động vật một khoảng được ghi rõ ở ô trên.

Lý do nằm ở bản chất của đối tượng. Phương tiện là vật thể cứng, có cạnh thẳng, tỉ lệ ổn định, và
thường xuất hiện trên một loại nền đặc trưng. Động vật là vật thể biến dạng, xuất hiện ở vô số tư
thế, và nhiều lớp trong nhóm này chia sẻ cùng một sơ đồ hình thể bốn chân. Mọi số liệu trong mục
này đều nhất quán với phân tích định tính đã nêu ở notebook 00 trước khi có bất kỳ mô hình nào
được huấn luyện.

In [16]:
hce = res_best['high_conf_errors']
fig, axes = plt.subplots(2, 4, figsize=(13.5, 7.6))
for ax, item in zip(axes.ravel(), hce):
    i = item['index']
    ax.imshow(x_test_raw[i])
    ax.set_title(f"#{i} · thật = {CLASS_VI[item['true']]} · dự đoán = {CLASS_VI[item['pred']]}\n"
                 f"độ tin cậy = {item['confidence']*100:.2f}%", fontsize=10.5, color='#B03A2E')
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.ravel()[len(hce):]:
    ax.axis('off')
fig.suptitle(f'Tám ảnh bị phân loại sai với độ tin cậy cao nhất ({best_name})\n'
             'trên 10 000 ảnh kiểm thử CIFAR-10', fontsize=14, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f'{FIG_DIR}/fig_cifar10_high_conf_errors.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_cifar10_high_conf_errors.png')
print()
print(f"{'STT':>4}{'Chỉ số ảnh':>12}{'Nhãn thật':>12}{'Dự đoán':>12}{'Độ tin cậy':>13}")
print('-' * 55)
for r, item in enumerate(hce, 1):
    print(f"{r:>4}{item['index']:>12}{CLASS_VI[item['true']]:>12}{CLASS_VI[item['pred']]:>12}"
          f"{item['confidence']*100:>12.2f}%")
print('-' * 55)
conf_all = res_best['probs'].max(axis=1)
pred_all = res_best['pred']
wrong_mask = pred_all != y_te
print(f'Độ tin cậy trung bình khi dự đoán ĐÚNG : {conf_all[~wrong_mask].mean()*100:.2f}%')
print(f'Độ tin cậy trung bình khi dự đoán SAI  : {conf_all[wrong_mask].mean()*100:.2f}%')
print(f'Số ảnh sai có độ tin cậy > 99%         : {int(((conf_all > 0.99) & wrong_mask).sum()):,}')
print(f'Tổng số ảnh sai                        : {int(wrong_mask.sum()):,}')
print(f'Tỉ lệ ảnh sai nhưng rất tự tin (>99%)  : '
      f'{100*((conf_all > 0.99) & wrong_mask).sum()/max(1, wrong_mask.sum()):.2f}% số ảnh sai')

Đã lưu fig_cifar10_high_conf_errors.png

 STT  Chỉ số ảnh   Nhãn thật     Dự đoán   Độ tin cậy
-------------------------------------------------------
   1        5416      xe tải        ô tô       99.92%
   2        2405         mèo         ếch       99.64%
   3        6434     máy bay    tàu thủy       99.58%
   4        2226        chim         ếch       99.55%
   5         802        hươu        chim       99.47%
   6        7892        ô tô      xe tải       99.18%
   7        4474     máy bay    tàu thủy       98.78%
   8        6979         chó        hươu       98.77%
-------------------------------------------------------
Độ tin cậy trung bình khi dự đoán ĐÚNG : 81.84%
Độ tin cậy trung bình khi dự đoán SAI  : 54.04%
Số ảnh sai có độ tin cậy > 99%         : 6
Tổng số ảnh sai                        : 2,242
Tỉ lệ ảnh sai nhưng rất tự tin (>99%)  : 0.27% số ảnh sai


**Diễn giải hình `fig_cifar10_high_conf_errors.png`.** Tám ảnh này là những trường hợp mô hình sai
mà vẫn gán xác suất rất cao cho nhãn sai — loại lỗi nguy hiểm nhất trong thực tế, vì độ tin cậy
cao thường được dùng làm căn cứ để bỏ qua khâu kiểm tra của con người.

Quan sát trực tiếp trên ảnh cho thấy phần lớn thuộc ba dạng: đối tượng bị cắt cụt hoặc chỉ thấy
một phần, đối tượng nhỏ trên nền chiếm ưu thế, và các cặp lớp gần nhau về hình thể. Một số ảnh khó
tới mức người xem cũng phải do dự, điều này phản ánh giới hạn thực sự của độ phân giải
$32 \times 32$ chứ không phải khiếm khuyết riêng của mô hình.

Hai con số ở cuối ô mã có ý nghĩa về hiệu chuẩn xác suất: độ tin cậy trung bình khi đúng cao hơn
rõ rệt so với khi sai, nghĩa là điểm tin cậy của mô hình **có** mang thông tin và dùng được làm
tín hiệu lọc. Tuy nhiên tỉ lệ ảnh sai mà vẫn tự tin trên 99% khác 0, nên ngưỡng tin cậy không bao
giờ là bảo đảm tuyệt đối.

## 11. Ghi tệp metrics tổng hợp

In [17]:
def pack_fw(res, hist, best_ep, ttime, n_params, framework, epochs, device):
    return {
        'framework': framework,
        'device': device,
        'params': int(n_params),
        'train_time_s': float(ttime),
        'epochs': int(epochs),
        'best_epoch': int(best_ep),
        'accuracy': res['accuracy'],
        'macro_precision': res['macro_precision'],
        'macro_recall': res['macro_recall'],
        'macro_f1': res['macro_f1'],
        'loss': res['loss'],
        'history': {k: [float(v) for v in hist[k]]
                    for k in ('train_loss', 'val_loss', 'train_acc', 'val_acc')},
        'confusion_matrix': res['confusion_matrix'],
        'per_class_accuracy': res['per_class_accuracy'],
        'high_conf_errors': res['high_conf_errors'],
    }


def pack_np(block):
    '''Chuyển khối NumPy từ notebook 01 sang lược đồ cuối, bảo đảm có khóa device.'''
    out = {k: v for k, v in block.items()}
    out.setdefault('device', 'cpu')
    return out


notes = (
    'THIET BI. Ba nhom mo hinh chay tren ba cau hinh phan cung khac nhau va dieu nay phai duoc '
    f'tinh den khi doc cot thoi gian: pytorch chay GPU ({GPU_NAME}, CUDA '
    f'{torch.version.cuda}); tensorflow chay CPU vi TensorFlow tu ban 2.11 da bo ho tro GPU '
    'native tren Windows; numpy_baseline va numpy_improved chay CPU vi NumPy la thu vien tinh '
    'toan tren CPU, tuc la ban chat cua de bai chu khong phai han che phan cung. He qua: cot '
    'train_time_s LA PHEP SO SANH PHAN CUNG, KHONG PHAI PHEP SO SANH KHUNG THU VIEN, va bao cao '
    'khong ket luan khung nao nhanh hon khung nao. Cac chi so chat luong (accuracy, macro '
    'precision/recall/F1, per_class_accuracy) khong phu thuoc thiet bi nen van so sanh duoc. Moi '
    'phep do thoi gian cua PyTorch deu boc torch.cuda.synchronize() o ca hai dau de khong do nham '
    'thoi gian xep hang lenh. '
    'QUY MO. Hai mo hinh framework dung DAY DU 40000 anh train va 10000 anh validation, '
    f'{EPOCHS_FW} epoch, batch {BATCH_FW}. Hai mo hinh NumPy thuan dung TAP CON PHAN TANG '
    f'{sub["n_train_subset"]} anh train va {sub["n_val_subset"]} anh validation '
    f'({EPOCHS_NP} epoch, batch {sub["batch_size"]}), xem khoa numpy_subset. CA BON mo hinh deu '
    'duoc danh gia tren tron ven 10000 anh cua tap kiem thu goc nen phep so sanh cong bang o phia '
    'danh gia; chenh lech giua nhom NumPy va nhom framework la tong hop cua ba yeu to: luong du '
    'lieu huan luyen (gap 4 lan), chinh quy hoa (framework co Dropout va BatchNorm, NumPy khong '
    'co) va chieu sau (3 khoi so voi 2). Viec tach bach ba yeu to do duoc lam rieng o '
    'analysis/notebooks/06_ablation_scaling.ipynb. '
    'KET QUA AM TINH. Tren tap con 10000 anh, bien the Improved KHONG vuot Baseline ve do chinh '
    f'xac ({np_impr["accuracy"]*100:.2f}% so voi {np_base["accuracy"]*100:.2f}%, chenh '
    f'{(np_impr["accuracy"]-np_base["accuracy"])*100:+.2f} diem phan tram, tuong duong '
    f'{abs(np_impr["accuracy"]-np_base["accuracy"])*10000:.0f} anh tren 10000) va con qua khop '
    f'nang hon (mat mat kiem thu {np_impr["loss"]:.4f} so voi {np_base["loss"]:.4f}). Ket qua nay '
    'duoc giu nguyen nhu da do; phan dien giai da duoc viet lai thay vi sua thuc nghiem. '
    'LUOC DO. Khoa roc_auc cua schema nhi phan khong ap dung cho bai toan 10 lop nen duoc luoc '
    'bo, thay bang macro_precision / macro_recall / macro_f1 / per_class_accuracy / '
    'high_conf_errors dung theo bien the da lop cua muc 5.3. Kiem chung gradient bang sai phan '
    f'huu han trung tam (eps=1e-5, float64) tren {partial["gradient_check"]["n_checks"]} vi tri '
    f'tham so cua mang BA KENH cho sai so tuong doi lon nhat '
    f'{partial["gradient_check"]["max_rel_error"]:.3e}. Chuan hoa dung ba cap hang so rieng cho ba '
    'kenh R, G, B, tich luy o float64 (tich luy o float32 tren 51 trieu gia tri lam trung binh '
    'bao hoa va tra ve ba kenh trung nhau). '
    'NGAN SACH. Tong thoi gian chay notebook 02 vuot nguong 20 phut cua muc 1 hop dong vi rieng '
    f'nhanh Keras tren CPU da ton khoang {time_tf/60:.0f} phut cho {EPOCHS_FW} epoch, trong khi '
    f'nhanh PyTorch tren GPU chi ton {time_pt/60:.1f} phut cho cung so epoch. Hai khung duoc cap '
    'cung mot ngan sach epoch de chi so chat luong so sanh duoc; day la mot sai lech co y thuc so '
    'voi hop dong, duoc ghi lai o day thay vi cat bot epoch cua Keras.'
)

metrics = {
    'domain': 'cifar10',
    'task': 'classification',
    'dataset': {
        'file': 'cifar10/data/cifar10.npz',
        'n_raw': int(len(x_train_raw) + len(x_test_raw)),
        'n_clean': int(len(x_train_raw) + len(x_test_raw)),
        'n_train': int(len(X_tr)),
        'n_val': int(len(X_va)),
        'n_test': int(len(X_te)),
        'n_features': 3072,
        'image_shape': [32, 32, 3],
        'n_classes': 10,
        'class_names_en': CLASS_EN,
        'class_names_vi': CLASS_VI,
        'preprocess': {'mean': [float(v) for v in MEAN_C],
                       'std': [float(v) for v in STD_C], 'scale': 255.0},
        'numpy_subset': {'n_train': int(sub['n_train_subset']),
                         'n_val': int(sub['n_val_subset'])},
    },
    'numpy_subset': {'n_train': int(sub['n_train_subset']),
                     'n_val': int(sub['n_val_subset']),
                     'n_test': int(sub['n_test']),
                     'epochs': EPOCHS_NP,
                     'batch_size': int(sub['batch_size']),
                     'device': 'cpu',
                     'reason': 'NumPy thuần chạy CPU theo định nghĩa của đề bài; '
                               'tập con giữ notebook 01 trong ngân sách 20 phút'},
    'devices': {
        'pytorch': {'device': DEV_PT, 'name': GPU_NAME,
                    'cuda': torch.version.cuda, 'torch': torch.__version__},
        'tensorflow': {'device': DEV_TF, 'name': 'CPU',
                       'reason': 'TensorFlow >= 2.11 bỏ hỗ trợ GPU native trên Windows',
                       'tensorflow': tf.__version__, 'keras': keras.__version__},
        'numpy': {'device': 'cpu', 'name': 'CPU', 'numpy': np.__version__},
        'timing_caveat': 'train_time_s so sánh phần cứng, không so sánh khung thư viện',
    },
    'models': {
        'numpy_baseline': pack_np(np_base),
        'numpy_improved': pack_np(np_impr),
        'pytorch':    pack_fw(res_pt, hist_pt, best_ep_pt, time_pt, n_pt,
                              f'PyTorch {torch.__version__}', EPOCHS_FW, DEV_PT),
        'tensorflow': pack_fw(res_tf, hist_tf, best_ep_tf, time_tf, n_tf,
                              f'TensorFlow {tf.__version__} / Keras {keras.__version__}',
                              EPOCHS_FW, DEV_TF),
    },
    'best_model': 'pytorch' if res_pt['accuracy'] >= res_tf['accuracy'] else 'tensorflow',
    'gradient_check': partial['gradient_check'],
    'notes': notes,
}

out = os.path.join(REP_DIR, 'metrics_cifar10.json')
with open(out, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print('Đã ghi', out, f'({os.path.getsize(out)/1024:.1f} KB)')
print()
for key, m in metrics['models'].items():
    assert len(m['confusion_matrix']) == 10 and len(m['confusion_matrix'][0]) == 10
    assert len(m['per_class_accuracy']) == 10
    assert len(m['high_conf_errors']) <= 8
    assert m['device'] in ('cuda', 'cpu'), f'khóa device sai ở {key}'
    print(f"  {key:16s} [{m['device']:>4s}] acc={m['accuracy']:.6f} "
          f"macroF1={m['macro_f1']:.6f} params={m['params']:>9,} "
          f"time={m['train_time_s']:8.1f}s epochs={m['epochs']:>2} "
          f"cfm=10x10 per_class=10 hce={len(m['high_conf_errors'])}")
print()
print('Mô hình tốt nhất ghi trong metrics :', metrics['best_model'])
print('Khối numpy_subset ở cấp cao nhất   :',
      f"{metrics['numpy_subset']['n_train']:,} train / "
      f"{metrics['numpy_subset']['n_val']:,} val, {metrics['numpy_subset']['epochs']} epoch")
print('Khối gradient_check ở cấp cao nhất :',
      f"{metrics['gradient_check']['n_checks']} phép kiểm, "
      f"sai số lớn nhất {metrics['gradient_check']['max_rel_error']:.3e}")
print('Thiết bị từng mô hình              :',
      {k: v['device'] for k, v in metrics['models'].items()})

Đã ghi ../reports\metrics_cifar10.json (30.7 KB)

  numpy_baseline   [ cpu] acc=0.569400 macroF1=0.565409 params=  153,962 time=   865.2s epochs=12 cfm=10x10 per_class=10 hce=8
  numpy_improved   [ cpu] acc=0.569100 macroF1=0.569730 params=  268,650 time=   271.6s epochs=12 cfm=10x10 per_class=10 hce=8
  pytorch          [cuda] acc=0.775800 macroF1=0.774022 params=  188,970 time=    48.5s epochs=25 cfm=10x10 per_class=10 hce=8
  tensorflow       [ cpu] acc=0.750200 macroF1=0.746899 params=  188,970 time=   475.2s epochs=25 cfm=10x10 per_class=10 hce=8

Mô hình tốt nhất ghi trong metrics : pytorch
Khối numpy_subset ở cấp cao nhất   : 10,000 train / 2,500 val, 12 epoch
Khối gradient_check ở cấp cao nhất : 15 phép kiểm, sai số lớn nhất 1.614e-10
Thiết bị từng mô hình              : {'numpy_baseline': 'cpu', 'numpy_improved': 'cpu', 'pytorch': 'cuda', 'tensorflow': 'cpu'}


In [18]:
required = [
    'fig_cifar10_class_distribution.png', 'fig_cifar10_sample_grid.png',
    'fig_cifar10_scratch_curves.png', 'fig_cifar10_scratch_confusion.png',
    'fig_cifar10_scratch_comparison.png', 'fig_cifar10_framework_curves.png',
    'fig_cifar10_framework_confusion.png', 'fig_cifar10_3way_benchmark.png',
    'fig_cifar10_per_class_accuracy.png', 'fig_cifar10_high_conf_errors.png',
]
print('Kiểm tra đủ 10 hình bắt buộc theo mục 6 hợp đồng tích hợp:')
ok = True
for f in required:
    p = os.path.join(FIG_DIR, f)
    e = os.path.exists(p)
    ok &= e
    print(f"  {'OK   ' if e else 'THIẾU'} {f:42s} "
          f"{os.path.getsize(p)/1024 if e else 0:8.1f} KB")
print()
print('Đủ 10/10 hình:', ok)
print()
print('Hiện vật mô hình bàn giao cho mlp_vs_cnn:')
for f in ['cifar10_cnn_pytorch.pt', 'cifar10_cnn_def.py', 'cifar10_preproc.json']:
    p = os.path.join(MODEL_DIR, f)
    print(f'  {f:26s} {os.path.getsize(p)/1024:8.1f} KB  tồn tại={os.path.exists(p)}')
print()
print('Tệp metrics:')
for f in ['metrics_cifar10.json', 'metrics_cifar10_scratch_partial.json',
          'cifar10_eda_summary.json']:
    p = os.path.join(REP_DIR, f)
    print(f'  {f:40s} {os.path.getsize(p)/1024:8.1f} KB  tồn tại={os.path.exists(p)}')

Kiểm tra đủ 10 hình bắt buộc theo mục 6 hợp đồng tích hợp:
  OK    fig_cifar10_class_distribution.png             88.0 KB
  OK    fig_cifar10_sample_grid.png                   472.6 KB
  OK    fig_cifar10_scratch_curves.png                175.9 KB
  OK    fig_cifar10_scratch_confusion.png             168.1 KB
  OK    fig_cifar10_scratch_comparison.png             71.0 KB
  OK    fig_cifar10_framework_curves.png              204.2 KB
  OK    fig_cifar10_framework_confusion.png           168.2 KB
  OK    fig_cifar10_3way_benchmark.png                 69.0 KB
  OK    fig_cifar10_per_class_accuracy.png             78.4 KB
  OK    fig_cifar10_high_conf_errors.png              128.9 KB

Đủ 10/10 hình: True

Hiện vật mô hình bàn giao cho mlp_vs_cnn:
  cifar10_cnn_pytorch.pt        746.7 KB  tồn tại=True
  cifar10_cnn_def.py              3.2 KB  tồn tại=True
  cifar10_preproc.json            1.3 KB  tồn tại=True

Tệp metrics:
  metrics_cifar10.json                         30.7 KB  tồn tại=True

## 12. Kết luận của miền CIFAR-10

Ba notebook của miền này đã hoàn thành trọn vẹn yêu cầu của đề bài đối với dữ liệu ảnh màu: cùng
một bài toán được giải bằng ba cách cài đặt độc lập, trên cùng một phép chia dữ liệu và cùng một
phép chuẩn hóa, rồi được đối chiếu trên cùng 10 000 ảnh kiểm thử.

**Về mặt kỹ thuật**, báo cáo đã chứng minh mạng tích chập viết tay xử lý đúng ảnh ba kênh, thông
qua kiểm chứng gradient bằng sai phân hữu hạn ở độ chính xác `float64` và qua việc đối chiếu với
một cài đặt tham chiếu viết bằng vòng lặp tường minh. Đây là điều kiện cần trước khi bất kỳ con số
độ chính xác nào được coi là có ý nghĩa.

**Về mặt thực nghiệm**, thứ tự kết quả nhất quán với kỳ vọng lý thuyết: Baseline thấp nhất,
Improved cao hơn nhờ gói đệm biên cộng He Normal cộng lịch learning rate, hai framework cao nhất
nhờ kiến trúc sâu hơn, có chính quy hóa và được huấn luyện trên gấp bốn lần dữ liệu. Hai framework
gần như ngang nhau, xác nhận rằng lựa chọn công cụ không quyết định chất lượng.

**Về mặt so sánh liên miền**, khoảng cách giữa kết quả CIFAR-10 và kết quả MNIST của cùng những
kiến trúc này chính là đóng góp phân tích của miền dữ liệu này cho báo cáo tổng thể: nó cho thấy
kết luận "CNN giải quyết tốt bài toán ảnh" cần được phát biểu thận trọng hơn, gắn với độ phức tạp
thị giác của dữ liệu cụ thể.

Ba hiện vật mô hình đã được lưu và kiểm chứng quy trình nạp lại, sẵn sàng cho notebook
`mlp_vs_cnn` phân tích không gian ẩn 128 chiều bằng PCA.